# Phát hiện, đếm và phân loại tươi/hỏng: chuối, táo, cam (hai tầng YOLO11, Google Colab)

```
Ảnh / frame nhiều quả
   │
   ▼
[Tầng 1] YOLO11 detection – 3 class: banana, apple, orange      (dữ liệu có box)
   │  cắt (crop) từng quả theo box
   ▼
[Tầng 2] YOLO11-cls – 2 class: fresh, rotten                    (dữ liệu phân loại, không cần box)
   │
   ▼
box + loại quả + tình trạng + số đếm theo loại
```

Notebook chạy **tuần tự từ trên xuống** trên Colab (GPU T4). Chỉ cần sửa **ô CONFIG**.

| Phần | Nội dung |
|---|---|
| 0 | Cấu hình, môi trường, định nghĩa class, hàm dùng chung |
| 1 | Tiền xử lý: tải Kaggle (tuỳ chọn) → tách frame → gộp dữ liệu detection (YOLO/VOC → YOLO 3 class) → audit → chia tập theo nhóm → dữ liệu classifier (thư mục theo class + crop từ ảnh tự quay) |
| 2 | Train baseline cho cả hai tầng |
| 3 | Đánh giá: `evaluate_det()` (mAP), `evaluate_cls()` (accuracy/F1/confusion), `evaluate_e2e()` (cả pipeline trên ảnh tự quay, gồm sai số đếm) |
| 4 | Fine-tune hai tầng bằng dữ liệu tự quay + replay public, so sánh trước/sau |
| 5 | Xuất model (`.pt`, ONNX), tải về, in phiên bản thư viện |

**Test độc lập** cho tầng 2 và end-to-end là ảnh/frame **tự quay**. Mọi kết quả ghi vào Google Drive. Notebook **không chứa số liệu mẫu**.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Không chạy trên Colab: bỏ qua bước mount Drive (dữ liệu nằm cạnh notebook).")

: 

## Phần 0. Cấu hình và môi trường

### 0.1 CONFIG (ô duy nhất cần sửa)
Các cờ `OVERWRITE[...]` mặc định `False`: ô nào ghi vào thư mục đã có dữ liệu sẽ **bỏ qua** thay vì ghi đè.

In [ ]:
# ======================= CONFIG: chỉ cần sửa ô này =======================
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- Đường dẫn ---
DRIVE_ROOT    = Path("/content/drive/MyDrive/fruit") if IN_COLAB else Path.cwd()   # gốc dữ liệu (Drive, hoặc thư mục chạy notebook)
LOCAL_DATASET = Path("/content/dataset") if IN_COLAB else DRIVE_ROOT / "local_dataset"   # bản sao cục bộ (đọc nhanh hơn Drive)

# --- Tầng 1: detection 3 class ---
MODEL_SIZE    = "yolo11n"
IMGSZ         = 640
EPOCHS_BASE   = 50
EPOCHS_FT     = 30
JOINT_MODEL   = "yolo11s"   # train JOINT (public + ảnh tự quay nhân bản): tốt hơn baseline -> fine-tune trên test_own (xem bàn giao 2026-10-18)
EPOCHS_JOINT  = 80
OWN_REP       = 4           # số lần nhân bản ảnh tự quay trong train joint
BATCH         = 16
OPTIMIZER     = "SGD"       # đặt cố định: với optimizer="auto" Ultralytics sẽ BỎ QUA lr0
LR0_BASE      = 0.01
LR0_FT        = 0.002       # fine-tune: nhỏ hơn baseline
FREEZE_FT     = False       # True -> đóng băng FREEZE_LAYERS lớp đầu khi fine-tune detection
FREEZE_LAYERS = 10

# --- Tầng 2: classifier fresh/rotten ---
CLS_MODEL     = f"{MODEL_SIZE}-cls"
CLS_IMGSZ     = 224
EPOCHS_CLS    = 30
EPOCHS_CLS_FT = 15
CLS_BATCH     = 64
CLS_OPTIMIZER = "AdamW"     # đặt cố định để lr0 có hiệu lực
LR0_CLS       = 0.001
LR0_CLS_FT    = 0.0002
CROP_PAD      = 0.10        # nới box thêm 10% mỗi phía khi cắt crop (cả lúc tạo dữ liệu lẫn lúc suy luận)
CROP_MIN_PX   = 24          # bỏ crop có cạnh nhỏ hơn 24 px khi tạo dữ liệu classifier

# --- Chung ---
SEED         = 42
PATIENCE     = 20           # dừng sớm nếu val không cải thiện sau 20 epoch
REPLAY_RATIO = 0.3          # tỷ lệ dữ liệu public (tập train) trộn vào train fine-tune để tránh quên
HSV_H, HSV_S, HSV_V = 0.015, 0.7, 0.4   # augmentation độ sáng / màu

# --- Tách frame video ---
FRAME_STEP     = 20
BLUR_THRESHOLD = 100.0      # Laplacian variance dưới ngưỡng -> frame mờ, bỏ

# --- Tự gán nhãn frame video (ô 1.1b): không cần gán tay ---
# Box = detector COCO huấn luyện sẵn (có banana/apple/orange), chỉ giữ đúng loại quả của thư mục video.
# Tình trạng = vị trí frame trong video (video quả hỏng dần theo thời gian): đầu -> fresh, cuối -> rotten, giữa -> bỏ.
AUTO_LABEL        = True          # False = không tự gán, dùng own/ do bạn gán tay
AUTO_DET_MODEL    = "yolo11l.pt"  # tải tự động
AUTO_CONF         = 0.15          # thấp vì quả hỏng thường bị detector COCO chấm điểm thấp
AUTO_MIN_BOX_FRAC = 0.05          # bỏ box nhỏ hơn 5% diện tích ảnh (logo, ảnh nhỏ trong màn hình cuối video, nho/ngón tay...)
AUTO_BATCH        = 4             # batch khi tự gán nhãn bằng yolo11l (16 sẽ hết VRAM trên GPU 6GB; T4 16GB chạy được 16)
AUTO_KEEP_LARGEST = True          # video quay MỘT vật thể: chỉ giữ box lớn nhất mỗi frame (COCO hay trả thêm box con/box thừa)
AUTO_VERIFY_CLS   = True          # kiểm chứng nhãn tình trạng bằng cls_baseline (chỉ train trên dữ liệu public): lệch thì bỏ frame
AUTO_VERIFY_CONF  = 0.8           # classifier phải đồng ý với nhãn theo vị trí frame với xác suất >= ngưỡng này
AUTO_FRESH_FRAC   = 0.25          # 25% đầu video -> fresh
AUTO_ROTTEN_FRAC  = 0.35          # 35% cuối video -> rotten
# Ghi đè theo video: khoá là một đoạn trong "<loại_quả>_<tên_video>" (không phân biệt hoa thường),
# giá trị "fresh" | "rotten" | "skip" | "timeline" (mặc định). Ví dụ: {"banana_time_lapse": "skip"}
AUTO_STATE_BY_VIDEO = {}

# --- Ngưỡng cảnh báo ít mẫu ---
MIN_BOX_TRAIN  = 50         # detection, tập train: >= 50 box/class
MIN_BOX_EVAL   = 10         # detection, val/test: >= 10 box/class
MIN_CROP_TRAIN = 100        # classifier, tập train: >= 100 ảnh mỗi (loại quả, tình trạng)
MIN_CROP_EVAL  = 30         # classifier, test: >= 30 ảnh mỗi (loại quả, tình trạng)

# --- Chia tập theo nhóm ảnh (tỷ lệ theo số box / số ảnh) ---
OWN_SPLIT_MODE   = "block"  # ảnh tự quay: "block" = chia theo KHỐI frame liên tiếp (mọi video/loại quả đều có ở train/val/test)
                            #              "video" = cả video là một nhóm (test là video CHƯA thấy; ít video thì test thiếu loại quả)
OWN_BLOCK_FRAMES = 8        # mode "block": số frame liên tiếp (đã lấy mẫu) của một video dính chung một tập
CLS_BALANCE_RATIO = 2.0     # classifier: mỗi loại quả, tình trạng nhiều ảnh hơn bị giảm xuống <= RATIO x tình trạng ít ảnh (train, ngẫu nhiên theo SEED)
SPLIT_RATIOS   = {"train": 0.70, "val": 0.15, "test": 0.15}
DUP_HASH_DIST  = 6          # dHash 64-bit cách nhau <= 6 bit -> cùng nhóm (tăng = gom mạnh tay hơn)
MAX_GROUP_FRAC = 0.25       # cảnh báo nếu một nhóm chiếm > 25% dữ liệu của nhóm nguồn

# --- Tải dataset Kaggle (tuỳ chọn; để {} nếu tự đặt dataset vào Drive) ---
# Dạng {"tên_thư_mục": "chủ_sở_hữu/slug-dataset"} (lấy từ URL kaggle.com/datasets/<chủ>/<slug>)
KAGGLE_DET = {}             # dataset có box (YOLO hoặc Pascal VOC)  -> raw/
KAGGLE_CLS = {}             # dataset phân loại theo thư mục         -> raw_cls/

# --- Đánh giá & phân tích lỗi ---
CONF_PRED       = 0.25      # ngưỡng confidence của detector khi chạy pipeline / tìm ảnh lỗi
IOU_MATCH       = 0.5       # IoU tối thiểu để ghép box dự đoán với box thật
LOW_CONF        = 0.5       # dự đoán đúng nhưng conf < 0.5 -> "độ tin cậy thấp"
N_ERROR_IMAGES  = 20
N_AUDIT_SAMPLES = 30
N_SPEED_IMAGES  = 100

# --- Cờ ghi đè (mặc định False = không ghi đè dữ liệu đã có) ---
OVERWRITE = {
    "frames":   False,      # work/frames_to_label/
    "merged":   False,      # work/merged_det/
    "audit":    False,      # work/audit/
    "split":    False,      # work/split_*.csv, work/data*.yaml
    "own_auto": False,      # own/ do ô 1.1b tự tạo (không bao giờ đụng own/ do bạn gán tay)
    "det_baseline": False, "det_finetune": False, "det_joint": False,     # runs/det_*
    "cls_baseline": False, "cls_finetune": False,     # runs/cls_*
}

# --- Tiếp tục train khi Colab ngắt phiên (chỉ bật khi cần) ---
RESUME = {"det_baseline": False, "det_finetune": False, "det_joint": False, "cls_baseline": False, "cls_finetune": False}

ULTRALYTICS_VERSION = ""    # "" = bản mới nhất; điền số phiên bản (vd lấy từ Phần 5) để cố định

# --- Đường dẫn suy ra (không cần sửa) ---
RAW, RAW_CLS = DRIVE_ROOT / "raw", DRIVE_ROOT / "raw_cls"
VIDEOS, OWN, CONFIGS = DRIVE_ROOT / "videos", DRIVE_ROOT / "own", DRIVE_ROOT / "configs"
WORK, RUNS, EXPORTS = DRIVE_ROOT / "work", DRIVE_ROOT / "runs", DRIVE_ROOT / "exports"
FRAMES_DIR   = WORK / "frames_to_label"
MERGED       = WORK / "merged_det"                  # dữ liệu detection đã gộp (YOLO 3 class)
AUDIT_DIR    = WORK / "audit"
SPLIT_IMAGES = WORK / "split_images.csv"            # detection: ảnh -> nhóm -> tập
SPLIT_REPORT = WORK / "split_report.csv"            # detection: danh sách nhóm/nguồn mỗi tập
SPLIT_CLS    = WORK / "split_cls.csv"               # classifier public: ảnh -> nhóm -> tập
DATA_YAML    = WORK / "data.yaml"
DATA_FT_YAML = WORK / "data_ft.yaml"
KAGGLE_JSON  = CONFIGS / "kaggle.json"              # token Kaggle của bạn trên Drive (KHÔNG đưa vào notebook)
KAGGLE_TOKEN = CONFIGS / "access_token"             # hoặc token kiểu mới KGAT_... (một dòng)
LOCAL_DET    = LOCAL_DATASET / "det"
LOCAL_CLS    = LOCAL_DATASET / "cls"
AUTO_VERIFY_WEIGHTS = RUNS / "cls_baseline" / "weights" / "best.pt"   # dùng cho AUTO_VERIFY_CLS
KAGGLE_DET = {"mbk": "mbkinaci/fruit-images-for-object-detection"}
KAGGLE_CLS = {"sriramr": "sriramr/fruits-fresh-and-rotten-for-classification"}
# --- Train lại 2026-10-18: thêm raw/rf_v7 (detect) và raw_cls/rf_v3i (classify) ---
for _k in ("merged", "audit", "split", "det_baseline", "det_finetune", "cls_baseline", "cls_finetune"):
    OVERWRITE[_k] = True


### 0.2 Kiểm tra GPU và cài Ultralytics
Ô này **dừng ngay** nếu chưa bật GPU (Runtime → Change runtime type → T4 GPU).

In [ ]:
import subprocess, sys
import torch

print("CUDA khả dụng:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Chưa bật GPU! Vào Runtime > Change runtime type > chọn T4 GPU rồi chạy lại."
print("GPU:", torch.cuda.get_device_name(0))

# Cài ultralytics (pip qua subprocess để ô vẫn là Python thuần)
pkg = "ultralytics" + (f"=={ULTRALYTICS_VERSION}" if ULTRALYTICS_VERSION else "")
import importlib.util
if ULTRALYTICS_VERSION or importlib.util.find_spec("ultralytics") is None:      # đã cài sẵn thì không cài lại
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import ultralytics
print("ultralytics:", ultralytics.__version__, "| torch:", torch.__version__)

### 0.3 Mount Google Drive, đặt seed, tạo thư mục

In [ ]:
import random
import numpy as np
import torch

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Seed cố định: chia tập, lấy mẫu replay và train lặp lại được
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Tạo thư mục đầu ra (không đụng tới raw/, raw_cls/, videos/, own/)
for d in (WORK, RUNS, EXPORTS):
    d.mkdir(parents=True, exist_ok=True)
for d in (RAW, RAW_CLS, VIDEOS, OWN, CONFIGS):
    print(f"{'[có]   ' if d.exists() else '[THIẾU]'} {d}")

### 0.4 Định nghĩa class (MỘT chỗ)
- Tầng 1 (detection): `FRUITS` = 3 loại quả.
- Tầng 2 (classifier): `STATES` = fresh / rotten.
- Kết quả cuối và nhãn ảnh tự quay: `CLASS_NAMES` = 6 tổ hợp `<loại_quả>_<tình_trạng>`; `split_class` tách tên thành (loại quả, tình trạng).

Mọi ô sau dùng chung các biến này.

In [ ]:
FRUITS = ["banana", "apple", "orange"]          # class detection: 0 banana, 1 apple, 2 orange
ND = len(FRUITS)
STATES = ["fresh", "rotten"]                     # class classifier (thư mục train/<state>/)
CLASS_NAMES = [f"{f}_{s}" for f in FRUITS for s in STATES]
# -> 0 banana_fresh, 1 banana_rotten, 2 apple_fresh, 3 apple_rotten, 4 orange_fresh, 5 orange_rotten
NC = len(CLASS_NAMES)
CLASS_TO_ID = {n: i for i, n in enumerate(CLASS_NAMES)}
FRUIT_TO_ID = {f: i for i, f in enumerate(FRUITS)}


def split_class(name):
    """'banana_rotten' -> ('banana', 'rotten')"""
    fruit, state = name.rsplit("_", 1)
    return fruit, state


def class6(fruit, state):
    return CLASS_TO_ID[f"{fruit}_{state}"]


CLASS_FRUIT = [split_class(n)[0] for n in CLASS_NAMES]
CLASS_STATE = [split_class(n)[1] for n in CLASS_NAMES]
assert set(CLASS_FRUIT) == set(FRUITS) and set(CLASS_STATE) == set(STATES)
print("Detection:", dict(enumerate(FRUITS)))
print("Classifier:", dict(enumerate(STATES)))
print("Kết quả cuối:", dict(enumerate(CLASS_NAMES)))

### 0.5 Hàm dùng chung
- Đọc/kiểm tra nhãn YOLO, đọc nhãn Pascal VOC (XML), cắt crop, vẽ box, hiển thị lưới ảnh, chép file song song.
- **Quy ước tên file**: dữ liệu đã gộp có dạng `<nguồn>__<tên_gốc>` (hai dấu gạch dưới). Nguồn = tên dataset public, hoặc `own-<loại_quả>_<video/buổi>` cho dữ liệu tự quay.
  Crop cho classifier có dạng `<nguồn>__<loại_quả>__<tên>.jpg` và nằm trong thư mục `<tình_trạng>/`.
- Gom **nhóm ảnh gần giống** (dHash + union-find) và gán trọn nhóm vào train/val/test để tránh rò rỉ.

In [ ]:
import os, re, json, time, shutil, hashlib, random
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor
import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SRC_SEP = "__"          # phân cách các phần trong tên file đã gộp
OWN_PREFIX = "own-"     # tiền tố nguồn của dữ liệu tự quay/chụp


def safe_name(s):
    """Chỉ giữ chữ, số, '-', '_' và không để '__' (dành cho SRC_SEP)."""
    s = re.sub(r"[^0-9A-Za-z\-_]+", "_", str(s))
    return re.sub(r"_+", "_", s).strip("_")


def source_of(filename):
    return Path(filename).name.split(SRC_SEP)[0]


def fruit_of_crop(filename):
    """Crop classifier: '<nguồn>__<loại_quả>__<tên>.jpg' -> loại quả."""
    return Path(filename).name.split(SRC_SEP)[1]


def is_own_source(src):
    return src.startswith(OWN_PREFIX)


def list_images(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)


def label_path_for(img_path):
    """Quy ước YOLO: .../images/.../a.jpg -> .../labels/.../a.txt (thay thư mục 'images' cuối cùng)."""
    parts = list(Path(img_path).parts)
    idx = [i for i, p in enumerate(parts) if p == "images"]
    if not idx:
        return Path(img_path).with_suffix(".txt")
    parts[idx[-1]] = "labels"
    return Path(*parts).with_suffix(".txt")


def find_data_yaml(root):
    """data.yaml ở gốc dataset, hoặc file nông nhất nếu dataset giải nén lồng thư mục."""
    root = Path(root)
    if (root / "data.yaml").exists():
        return root / "data.yaml"
    found = sorted(root.rglob("data.yaml"), key=lambda p: len(p.parts))
    return found[0] if found else None


def load_names(data_yaml):
    d = yaml.safe_load(Path(data_yaml).read_text()) or {}
    names = d.get("names")
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names, key=int)]
    if not names:
        raise ValueError(f"{data_yaml} không có khóa 'names'")
    return [str(n) for n in names]


def load_remap(remap_file, names, targets, problems, tag):
    """-> {tên_gốc: đích hoặc None (bỏ)}. Ghi mọi vấn đề vào `problems`, không đoán."""
    cfg = yaml.safe_load(Path(remap_file).read_text()) or {}
    mp = cfg.get("map")
    if not isinstance(mp, dict):
        problems.append(f"[{tag}] {Path(remap_file).name}: thiếu khóa 'map:' (xem README)")
        return None
    mp = {str(k): v for k, v in mp.items()}
    unmapped = [n for n in names if n not in mp]
    if unmapped:
        problems.append(f"[{tag}] CHƯA ÁNH XẠ trong {Path(remap_file).name}: {unmapped}\n"
                        + "    thêm vào mục map:\n" + "\n".join(f'      "{n}": ' for n in unmapped))
    wrong = {k: v for k, v in mp.items() if v is not None and v not in targets}
    if wrong:
        problems.append(f"[{tag}] đích ánh xạ không thuộc {targets}: {wrong}")
    extra = [k for k in mp if k not in names]
    if extra:
        print(f"[{tag}] lưu ý: remap có tên không gặp trong dữ liệu (bỏ qua): {extra}")
    return None if (unmapped or wrong) else {n: mp[n] for n in names}


def read_yolo_label(path, nc=None):
    """Đọc nhãn YOLO 'class x y w h' (chuẩn hoá 0-1).
    Trả về (boxes hợp lệ [(cls,x,y,w,h)], dòng lỗi [(số_dòng, nội_dung, lý_do)])."""
    boxes, bad = [], []
    path = Path(path)
    if not path.exists():
        return boxes, bad
    for ln, line in enumerate(path.read_text(errors="replace").splitlines(), 1):
        tok = line.split()
        if not tok:
            continue
        reason = None
        if len(tok) != 5:
            reason = f"có {len(tok)} giá trị (cần 5: class x y w h)"
        else:
            try:
                c = float(tok[0])
                x, y, w, h = map(float, tok[1:])
            except ValueError:
                reason = "giá trị không phải số"
            else:
                if c != int(c) or c < 0 or (nc is not None and c >= nc):
                    reason = f"class id '{tok[0]}' không hợp lệ" + (f" (cần 0-{nc - 1})" if nc else "")
                elif not all(0.0 <= v <= 1.0 for v in (x, y, w, h)) or w <= 0 or h <= 0:
                    reason = "toạ độ ngoài [0,1] hoặc w/h <= 0"
        if reason:
            bad.append((ln, line.strip(), reason))
        else:
            boxes.append((int(c), x, y, w, h))
    return boxes, bad


def read_voc(xml_path):
    """Pascal VOC XML -> (width, height, [(tên_class, xmin, ymin, xmax, ymax)]). width/height = 0 nếu XML thiếu <size>."""
    r = ET.parse(xml_path).getroot()
    size = r.find("size")
    w = int(float(size.findtext("width", "0"))) if size is not None else 0
    h = int(float(size.findtext("height", "0"))) if size is not None else 0
    objs = []
    for o in r.findall("object"):
        bb = o.find("bndbox")
        objs.append((o.findtext("name", "").strip(), *[float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax")]))
    return w, h, objs


def yolo_to_xyxy(boxes, w, h):
    """[(cls,x,y,bw,bh)] chuẩn hoá -> [(cls, x1,y1,x2,y2)] theo pixel."""
    return [(c, (x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h) for c, x, y, bw, bh in boxes]


def crop_box(img, xyxy, pad=CROP_PAD):
    """Cắt vùng box (nới thêm `pad` mỗi phía), luôn trả về ảnh không rỗng."""
    h, w = img.shape[:2]
    x1, y1, x2, y2 = xyxy
    px, py = (x2 - x1) * pad, (y2 - y1) * pad
    x1, y1 = int(max(0, min(w - 1, x1 - px))), int(max(0, min(h - 1, y1 - py)))
    x2, y2 = int(min(w, max(x1 + 1, x2 + px))), int(min(h, max(y1 + 1, y2 + py)))
    return img[y1:y2, x1:x2]


def draw_box(img, xyxy, text, color):
    h, w = img.shape[:2]
    t = max(1, round(max(h, w) / 400))
    x1, y1, x2, y2 = [int(round(v)) for v in xyxy]
    cv2.rectangle(img, (x1, y1), (x2, y2), color, t)
    cv2.putText(img, text, (x1, max(12 * t, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.4 * t, color, t, cv2.LINE_AA)


def show_grid(paths, ncols=5, title=None, cell=3.2, captions=None):
    """Hiển thị lưới ảnh (đọc bằng OpenCV)."""
    paths = list(paths)
    if not paths:
        print("(không có ảnh để hiển thị)")
        return
    nrows = (len(paths) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(cell * ncols, cell * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for k, (ax, p) in enumerate(zip(axes.ravel(), paths)):
        img = cv2.imread(str(p))
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(captions[k] if captions else Path(p).name[:40], fontsize=7)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()
    return fig


def parallel_map(fn, items, workers=16):
    """Chạy song song (Drive I/O chậm khi xử lý tuần tự từng file)."""
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(fn, items))


def ensure_fresh_dir(path, overwrite_key):
    """True nếu được phép ghi vào `path`. Thư mục đã có dữ liệu và OVERWRITE[key]=False -> bỏ qua (không xoá gì)."""
    path = Path(path)
    assert path.is_relative_to(WORK) or path.is_relative_to(LOCAL_DATASET), f"Chỉ được ghi trong {WORK} hoặc {LOCAL_DATASET}"
    if path.exists() and any(path.iterdir()):
        if not OVERWRITE[overwrite_key]:
            print(f"[BỎ QUA] {path} đã có dữ liệu. Đặt OVERWRITE['{overwrite_key}']=True trong CONFIG nếu muốn tạo lại.")
            return False
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return True


def write_text_guarded(path, text, overwrite_key):
    """Ghi file trong work/: nếu đã có với nội dung KHÁC thì cần OVERWRITE[key]=True."""
    path = Path(path)
    assert path.is_relative_to(WORK)
    if path.exists() and path.read_text() != text and not OVERWRITE[overwrite_key]:
        raise RuntimeError(f"{path} đã tồn tại với nội dung khác. Đặt OVERWRITE['{overwrite_key}']=True nếu muốn ghi đè.")
    path.write_text(text)


def fresh_local_dir(path):
    """Thư mục trên ổ cục bộ Colab: tạo lại mỗi lần (không phải dữ liệu gốc)."""
    path = Path(path)
    assert path.is_relative_to(LOCAL_DATASET)
    shutil.rmtree(path, ignore_errors=True)
    path.mkdir(parents=True)
    return path


def fingerprint(names):
    """Mã băm danh sách tên file: để chắc baseline và fine-tune được đánh giá trên CÙNG tập."""
    return hashlib.md5("\n".join(sorted(map(str, names))).encode()).hexdigest()


# ---------- Gom nhóm ảnh gần giống (chống rò rỉ khi chia tập) ----------
_RF = re.compile(r"^(.*)_(?:jpg|jpeg|png|bmp|webp)\.rf\.[0-9A-Za-z]+$", re.I)


def rf_base(stem):
    """Roboflow tạo nhiều bản augment của cùng một ảnh: 'abc_jpg.rf.<mã>' -> 'abc'. Không khớp -> None."""
    m = _RF.match(stem)
    return m.group(1) if m else None


def dhash_pair(path):
    """dHash 64-bit của ảnh và của ảnh lật ngang. Ảnh gần như phẳng/không đọc được -> None (không dùng để gom)."""
    g = cv2.imread(str(path), cv2.IMREAD_REDUCED_GRAYSCALE_4)
    if g is None:
        return None
    s = cv2.resize(g, (9, 8), interpolation=cv2.INTER_AREA)
    def h(a):
        return int.from_bytes(np.packbits((a[:, 1:] > a[:, :-1]).flatten()).tobytes(), "big")
    hs, hf = h(s), h(s[:, ::-1])
    if not 8 <= bin(hs).count("1") <= 56:
        return None
    return hs, hf


def cluster_images(paths, keys=None):
    """Gom ảnh thành cụm: cùng `keys[i]` (khác None) hoặc dHash cách nhau <= DUP_HASH_DIST (so cả bản lật).
    Trả về chỉ số gốc của cụm cho từng ảnh."""
    n = len(paths)
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
    if keys is not None:
        seen = {}
        for i, k in enumerate(keys):
            if k is None:
                continue
            if k in seen:
                union(i, seen[k])
            else:
                seen[k] = i
    hashes = parallel_map(dhash_pair, paths)
    ok = [(i, h) for i, h in enumerate(hashes) if h is not None]
    if ok:
        idx = np.array([i for i, _ in ok])
        H = np.array([h[0] for _, h in ok], dtype=np.uint64)
        HF = np.array([h[1] for _, h in ok], dtype=np.uint64)
        POP = np.array([bin(v).count("1") for v in range(256)], dtype=np.uint8)
        pc = lambda x: POP[np.ascontiguousarray(x).view(np.uint8).reshape(*x.shape, 8)].sum(-1)
        for s0 in range(0, len(idx), 256):
            a = H[s0:s0 + 256, None]
            d = np.minimum(pc(a ^ H[None, :]), pc(a ^ HF[None, :]))
            for x, y in zip(*np.nonzero(d <= DUP_HASH_DIST)):
                if s0 + x < y:
                    union(int(idx[s0 + x]), int(idx[y]))
    return [find(i) for i in range(n)]


def assign_groups(g, ratios, seed, cols, size_col):
    """g: DataFrame index=nhóm, có các cột đếm `cols` và cột kích thước `size_col`.
    Xáo nhóm theo SEED, sắp giảm dần theo kích thước, rồi gán trọn từng nhóm vào tập đang thiếu nhiều nhất (tính theo từng cột)."""
    rng = random.Random(seed)
    items = list(g.index)
    rng.shuffle(items)
    items.sort(key=lambda k: -g.at[k, size_col])      # sort ổn định: hoà nhau giữ thứ tự xáo theo SEED
    tot = g[cols].sum().to_numpy(float)
    tot_size = float(g[size_col].sum())
    cur = {k: np.zeros(len(cols)) for k in ratios}
    cur_size = {k: 0.0 for k in ratios}
    out = {}
    for grp in items:
        cnt = g.loc[grp, cols].to_numpy(float)
        if cnt.sum() > 0:
            score = {k: float((cnt * (ratios[k] * tot - cur[k]) / np.maximum(tot, 1)).sum()) for k in ratios}
        else:
            score = {k: ratios[k] * tot_size - cur_size[k] for k in ratios}
        best = max(score, key=score.get)
        out[grp] = best
        cur[best] += cnt
        cur_size[best] += g.at[grp, size_col]
    return out


def warn_big_groups(g, size_col, label):
    biggest = g[size_col].max() / max(1, g[size_col].sum())
    if len(g) < 3:
        print(f"⚠️  {label}: chỉ có {len(g)} nhóm, không đủ để có cả train/val/test.")
    if biggest > MAX_GROUP_FRAC:
        print(f"⚠️  {label}: nhóm lớn nhất chiếm {biggest:.0%} (> {MAX_GROUP_FRAC:.0%}): {g[size_col].idxmax()}. "
              "Tỷ lệ chia sẽ lệch; thử giảm DUP_HASH_DIST.")


def show_training(run_dir):
    """Hiển thị results.png và vẽ lại chỉ số chính từ results.csv của một lần train."""
    run_dir = Path(run_dir)
    png, csv = run_dir / "results.png", run_dir / "results.csv"
    if png.exists():
        display(IPImage(filename=str(png), width=1000))
    if not csv.exists():
        print("Chưa có", csv)
        return None
    df = pd.read_csv(csv)
    df.columns = [c.strip() for c in df.columns]
    cols = [c for c in ("metrics/mAP50(B)", "metrics/mAP50-95(B)", "metrics/accuracy_top1") if c in df.columns]
    if cols:
        df.plot(x="epoch", y=cols, figsize=(7, 3.5), grid=True, title=run_dir.name)
        plt.show()
    display(df.tail(5))
    return df


print("Đã định nghĩa hàm dùng chung.")

## Phần 1. Tiền xử lý dữ liệu

### 1.0 Tải dataset Kaggle (tuỳ chọn, dùng `kagglehub`)
Điền `KAGGLE_DET` (dataset có box → `raw/`) và `KAGGLE_CLS` (dataset phân loại theo thư mục → `raw_cls/`) trong CONFIG,
ví dụ `KAGGLE_DET = {"mbk": "mbkinaci/fruit-images-for-object-detection"}`.

Token Kaggle (Kaggle → Settings → API → Generate New Token), chọn **một** trong ba cách, notebook **không chứa token**:
1. **Colab Secrets** (khuyên dùng): biểu tượng 🔑 bên trái → Add new secret, tên `KAGGLE_API_TOKEN`, giá trị `KGAT_...`, bật Notebook access.
2. File `configs/access_token` trên Drive, chỉ một dòng `KGAT_...`.
3. File `configs/kaggle.json` kiểu cũ (`username`, `key`).

Dataset đã có trên Drive thì bỏ qua (không ghi đè). Tải xong, viết file remap theo tên class / tên thư mục được in ra (xem README).

In [ ]:
import subprocess, sys

todo = [(RAW, KAGGLE_DET), (RAW_CLS, KAGGLE_CLS)]
if not (KAGGLE_DET or KAGGLE_CLS):
    print("KAGGLE_DET và KAGGLE_CLS rỗng -> bỏ qua (tự đặt dataset vào Drive hoặc điền trong CONFIG).")
else:
    kdir = Path.home() / ".kaggle"
    kdir.mkdir(exist_ok=True)
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        pass                                          # chưa tạo secret hoặc không chạy trên Colab
    if not token and KAGGLE_TOKEN.exists():
        token = KAGGLE_TOKEN.read_text().strip()
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token        # chỉ nằm trong bộ nhớ phiên, không in ra
        (kdir / "access_token").write_text(token)
        os.chmod(kdir / "access_token", 0o600)
        print("Dùng token Kaggle kiểu mới (Secrets hoặc configs/access_token).")
    else:
        assert KAGGLE_JSON.exists(), (f"Không có token Kaggle: tạo secret KAGGLE_API_TOKEN, hoặc đặt {KAGGLE_TOKEN} "
                                      f"hay {KAGGLE_JSON} trên Drive (xem hướng dẫn phía trên).")
        shutil.copyfile(KAGGLE_JSON, kdir / "kaggle.json")
        os.chmod(kdir / "kaggle.json", 0o600)
        print("Dùng configs/kaggle.json.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub"], check=True)
    import kagglehub

    for base, mapping in todo:
        base.mkdir(parents=True, exist_ok=True)
        for name, slug in mapping.items():
            dest = base / safe_name(name)
            if dest.exists() and any(dest.iterdir()):
                print(f"[BỎ QUA] {dest} đã có dữ liệu (không ghi đè).")
                continue
            src = Path(kagglehub.dataset_download(slug))      # tải vào cache cục bộ của Colab, trả về thư mục đã giải nén
            shutil.copytree(src, dest)                         # rồi mới chép lên Drive
            imgs = list_images(dest)
            print(f"[{base.name}/{dest.name}] {len(imgs)} ảnh, {len(list(dest.rglob('*.txt')))} .txt, {len(list(dest.rglob('*.xml')))} .xml")
            dy = find_data_yaml(dest)
            if dy is not None:
                print("   class gốc (data.yaml):", load_names(dy))
            print("   thư mục chứa ảnh:", sorted({p.parent.name for p in imgs})[:30])

### 1.1 Tách frame từ video tự quay
Đọc `videos/<loại_quả>/*.mp4`, lấy 1 frame mỗi `FRAME_STEP` frame, bỏ frame mờ (phương sai Laplacian < `BLUR_THRESHOLD`).
Frame lưu vào `work/frames_to_label/` tên `<loại_quả>_<tên_video>_<số_frame>.jpg`.

> Frame **chưa có nhãn**. Ô 1.1b ngay dưới tự gán nhãn (mặc định). Muốn gán tay: đặt `AUTO_LABEL=False`, gán nhãn bên ngoài (Roboflow/CVAT), xuất YOLO, đặt vào `own/` (xem README). Nên gán **6 class** (`banana_fresh`…):
> notebook tự tách thành nhãn 3 class cho detection và crop fresh/rotten cho classifier. Ảnh chỉ gán 3 class (`banana`…) chỉ dùng được cho detection.
> Bảng in ra có trung vị Laplacian của mỗi video để bạn chỉnh `BLUR_THRESHOLD`.

In [ ]:
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv"}
videos = sorted(p for p in VIDEOS.glob("*/*") if p.suffix.lower() in VIDEO_EXTS) if VIDEOS.exists() else []
print(f"Tìm thấy {len(videos)} video trong {VIDEOS}")

# Có video mới chưa có frame nào -> tách lại và tạo lại own/ (ô 1.1b), khỏi phải đổi cờ bằng tay
_have = [p.name for p in FRAMES_DIR.glob("*.jpg")] if FRAMES_DIR.exists() else []
_missing = [vp.name for vp in videos if vp.parent.name in FRUITS
            and not any(n.startswith(f"{vp.parent.name}_{safe_name(vp.stem)}_") for n in _have)]
if _have and _missing:
    print(f"Phát hiện {len(_missing)} video chưa có frame (vd {_missing[0]}): tách lại toàn bộ và tạo lại own/ tự gán.")
    OVERWRITE["frames"] = OVERWRITE["own_auto"] = True

if videos and ensure_fresh_dir(FRAMES_DIR, "frames"):
    rows = []
    for vp in videos:
        fruit = vp.parent.name
        if fruit not in FRUITS:
            print(f"[BỎ QUA] {vp}: thư mục '{fruit}' không thuộc {FRUITS}")
            continue
        vname = safe_name(vp.stem)
        cap = cv2.VideoCapture(str(vp))
        idx = kept = blurry = 0
        laps = []
        while True:
            if not cap.grab():                 # grab() nhanh hơn read() khi bỏ qua frame
                break
            if idx % FRAME_STEP == 0:
                ok, frame = cap.retrieve()
                if ok:
                    # Độ sắc nét: phương sai của Laplacian trên ảnh xám
                    lap = cv2.Laplacian(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
                    laps.append(lap)
                    if lap < BLUR_THRESHOLD:
                        blurry += 1
                    else:
                        cv2.imwrite(str(FRAMES_DIR / f"{fruit}_{vname}_{idx:06d}.jpg"), frame)
                        kept += 1
            idx += 1
        cap.release()
        rows.append(dict(video=str(vp.relative_to(VIDEOS)), total_frames=idx, sampled=len(laps),
                         kept=kept, blurry_dropped=blurry,
                         lap_median=round(float(np.median(laps)), 1) if laps else None))
    rep = pd.DataFrame(rows)
    display(rep)
    print("Tổng frame giữ lại:", int(rep.kept.sum()) if len(rep) else 0, "->", FRAMES_DIR)

### 1.1b Tự gán nhãn 6 class cho frame video (không cần gán tay)
Với mỗi frame trong `work/frames_to_label/`:
- **Box**: detector COCO huấn luyện sẵn (`AUTO_DET_MODEL`), chỉ giữ box đúng loại quả của thư mục video (`videos/banana/` → chỉ `banana`).
- **Tình trạng**: theo vị trí frame trong video: `AUTO_FRESH_FRAC` đầu → `fresh`, `AUTO_ROTTEN_FRAC` cuối → `rotten`, đoạn giữa bỏ (mơ hồ). Video không theo quy luật này thì chỉnh `AUTO_STATE_BY_VIDEO`.

Kết quả ghi vào `own/` (YOLO 6 class + `data.yaml`), rồi bật `OVERWRITE` cho gộp/audit/chia để các ô sau dùng ngay dữ liệu mới.
Chỉ chạy khi `own/` trống hoặc do ô này tạo; **không bao giờ ghi đè `own/` bạn gán tay**.

> **Lưu ý khi đọc kết quả**: nhãn tự động là nhãn *yếu* (detector COCO + quy ước theo thời gian), nên chỉ số trên ảnh tự quay đo mức khớp với nhãn yếu này, không phải độ chính xác thực tế. Xem ảnh mẫu ở đầu ra để kiểm tra bằng mắt.
**Lọc nhãn tự động** (xem 5 tham số `AUTO_*` ở CONFIG): bỏ box nhỏ (logo, ảnh nhỏ ở màn hình cuối video), chỉ giữ box lớn nhất mỗi frame, và **kiểm chứng tình trạng bằng `cls_baseline`**
(frame bị bỏ nếu classifier không đồng ý với nhãn theo vị trí frame; cột `cls_reject`). Video quả hỏng thường lặp lại cảnh đầu ở cuối nên nhãn theo vị trí frame một mình là chưa đủ.
Vì `cls_baseline` chỉ train trên dữ liệu public nên không rò rỉ sang dữ liệu tự quay. **Nên mở ảnh mẫu bên dưới để kiểm tra bằng mắt.**


In [ ]:
COCO_ID = {"banana": 46, "apple": 47, "orange": 49}     # id lớp trong COCO
AUTO_FLAGGED = False


def auto_label_frames():
    """Tạo own/ (YOLO 6 class) từ frames_to_label. True nếu vừa ghi own/ mới."""
    from ultralytics import YOLO
    marker = OWN / ".auto_labeled"
    if not AUTO_LABEL:
        print("[BỎ QUA] AUTO_LABEL=False")
        return False
    frames = list_images(FRAMES_DIR)
    if not frames:
        print(f"[BỎ QUA] chưa có frame trong {FRAMES_DIR} (chạy ô 1.1 trước)")
        return False
    if list_images(OWN) or (OWN / "data.yaml").exists():       # chỉ ảnh/nhãn mới tính; thư mục video lạc vào own/ thì bỏ qua
        if not marker.exists():
            print(f"[BỎ QUA] {OWN} đã có dữ liệu do bạn đặt vào: giữ nguyên, không ghi đè.")
            return False
        if not OVERWRITE["own_auto"]:
            print(f"[BỎ QUA] {OWN} đã được tự gán nhãn. Đặt OVERWRITE['own_auto']=True trong CONFIG nếu muốn tạo lại.")
            return False
        shutil.rmtree(OWN / "images", ignore_errors=True)
        shutil.rmtree(OWN / "labels", ignore_errors=True)
        (OWN / "data.yaml").unlink(missing_ok=True)

    pat = re.compile(rf"^({'|'.join(FRUITS)})_(.+)_(\d+)$")
    vids, skipped = {}, 0
    for p in frames:
        m = pat.match(p.stem)
        if m:
            vids.setdefault((m[1], m[2]), []).append((int(m[3]), p))
        else:
            skipped += 1
    if skipped:
        print(f"Bỏ {skipped} frame có tên không đúng dạng <loại_quả>_<tên_video>_<số>")

    def state_of(fruit, vid, pos):
        key = f"{fruit}_{vid}".lower()
        mode = next((v for k, v in AUTO_STATE_BY_VIDEO.items() if k.lower() in key), "timeline")
        if mode in STATES:
            return mode
        if mode == "skip":
            return None
        if pos <= AUTO_FRESH_FRAC:
            return "fresh"
        if pos >= 1 - AUTO_ROTTEN_FRAC:
            return "rotten"
        return None

    if (OWN / "videos").exists():
        print(f"⚠️  {OWN / 'videos'} không được dùng: video phải nằm ở {VIDEOS}/<loại_quả>/ (có thể xoá thư mục này).")
    det = YOLO(AUTO_DET_MODEL)
    verifier = None
    if AUTO_VERIFY_CLS:
        if AUTO_VERIFY_WEIGHTS.exists():
            verifier = YOLO(str(AUTO_VERIFY_WEIGHTS))
            print(f"Kiểm chứng nhãn tình trạng bằng {AUTO_VERIFY_WEIGHTS.parent.parent.name} (đồng ý >= {AUTO_VERIFY_CONF}).")
        else:
            print("⚠️  Chưa có cls_baseline nên CHƯA kiểm chứng nhãn tình trạng (nhãn theo vị trí frame có thể sai ở video lặp/chạy ngược). "
                  "Train xong Phần 2, đặt OVERWRITE['own_auto']=True rồi chạy lại ô này.")
    (OWN / "images").mkdir(parents=True, exist_ok=True)
    (OWN / "labels").mkdir(parents=True, exist_ok=True)
    rows, kept = [], []
    for (fruit, vid), items in sorted(vids.items()):
        items.sort()
        last = max(i for i, _ in items) or 1
        st = dict(frames=len(items), mid_dropped=0, no_box=0, cls_reject=0, fresh=0, rotten=0)
        todo = []
        for idx, p in items:
            s = state_of(fruit, vid, idx / last)
            if s is None:
                st["mid_dropped"] += 1
            else:
                todo.append((p, s))
        if todo:
            def _predict_chunks():                    # tự chia lô: truyền cả danh sách đường dẫn thì ultralytics nạp gần hết một lượt (hết VRAM trên GPU nhỏ)
                for k in range(0, len(todo), AUTO_BATCH):
                    yield from det.predict([str(p) for p, _ in todo[k:k + AUTO_BATCH]], classes=[COCO_ID[fruit]], conf=AUTO_CONF,
                                           imgsz=IMGSZ, verbose=False)
            res = _predict_chunks()
            for (p, s), r in zip(todo, res):
                b = r.boxes.xywhn.cpu().numpy()
                xy = r.boxes.xyxy.cpu().numpy()
                big = b[:, 2] * b[:, 3] >= AUTO_MIN_BOX_FRAC
                b, xy = b[big], xy[big]
                if AUTO_KEEP_LARGEST and len(b) > 1:
                    k = int(np.argmax(b[:, 2] * b[:, 3]))
                    b, xy = b[k:k + 1], xy[k:k + 1]
                if not len(b):
                    st["no_box"] += 1
                    continue
                if verifier is not None:
                    img0 = cv2.imread(str(p))
                    pr = verifier.predict([crop_box(img0, q) for q in xy], imgsz=CLS_IMGSZ, verbose=False)
                    agree = [i for i, x in enumerate(pr) if verifier.names[int(x.probs.top1)] == s and float(x.probs.top1conf) >= AUTO_VERIFY_CONF]
                    b = b[agree]
                    if not len(b):
                        st["cls_reject"] += 1
                        continue
                cid = CLASS_TO_ID[f"{fruit}_{s}"]
                shutil.copyfile(p, OWN / "images" / p.name)
                (OWN / "labels" / (p.stem + ".txt")).write_text(
                    "".join(f"{cid} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n" for x, y, w, h in b))
                st[s] += 1
                kept.append((p, cid, b))
        rows.append(dict(video=f"{fruit}/{vid}"[:60], **st))

    rep = pd.DataFrame(rows)
    display(rep)
    if not kept:
        print("⚠️  Không frame nào có box: kiểm tra AUTO_CONF / AUTO_DET_MODEL. Không ghi own/.")
        shutil.rmtree(OWN / "images", ignore_errors=True)
        shutil.rmtree(OWN / "labels", ignore_errors=True)
        return False
    (OWN / "data.yaml").write_text(yaml.safe_dump({"path": str(OWN), "names": dict(enumerate(CLASS_NAMES))},
                                                  allow_unicode=True, sort_keys=False))
    marker.write_text("own/ này do ô 1.1b tự tạo; ô đó chỉ ghi đè khi OVERWRITE['own_auto']=True.\n")
    per = pd.Series([CLASS_NAMES[c] for _, c, b in kept for _ in range(len(b))]).value_counts().reindex(CLASS_NAMES, fill_value=0)
    print(f"Đã ghi {len(kept)} ảnh có nhãn vào {OWN}. Số box mỗi class:")
    print(per.to_string())
    for n, v in per.items():
        if v == 0:
            print(f"⚠️  {n}: 0 box (kiểm tra AUTO_FRESH_FRAC / AUTO_ROTTEN_FRAC / AUTO_STATE_BY_VIDEO)")

    pick = random.Random(SEED).sample(kept, min(12, len(kept)))
    fig, axs = plt.subplots(3, 4, figsize=(16, 10))
    for ax in axs.ravel():
        ax.axis("off")
    for ax, (p, cid, b) in zip(axs.ravel(), pick):
        im = cv2.imread(str(p))
        h, w = im.shape[:2]
        for x, y, bw, bh in b:
            cv2.rectangle(im, (int((x - bw / 2) * w), int((y - bh / 2) * h)),
                          (int((x + bw / 2) * w), int((y + bh / 2) * h)), (0, 255, 0), 4)
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{CLASS_NAMES[cid]}  ({p.name[:28]}...)", fontsize=9)
    plt.tight_layout()
    plt.show()
    return True


if auto_label_frames():
    if SPLIT_IMAGES.exists() and not (WORK / "split_images.prev.csv").exists():
        shutil.copyfile(SPLIT_IMAGES, WORK / "split_images.prev.csv")   # để ô sau 1.4 kiểm tra ảnh public không đổi tập
    for k in ("merged", "audit", "split"):
        OVERWRITE[k] = True
    AUTO_FLAGGED = True
    print("→ Đã bật OVERWRITE merged/audit/split để gộp và chia lại có dữ liệu tự quay (ô sau 1.4 sẽ tắt lại).")

### 1.2 Gộp dữ liệu detection: đổi sang YOLO 3 class
Đọc từng `raw/<dataset>/` và `own/`, đổi nhãn về **3 class** (`banana`, `apple`, `orange`), gộp vào `work/merged_det/` với tên `<nguồn>__<tên_gốc>`.
- **Định dạng nhận được**: YOLO (có `data.yaml` + `labels/*.txt`) hoặc **Pascal VOC** (file `.xml` cùng tên ảnh, ví dụ bộ mbkinaci) → tự đổi sang YOLO.
- **Public**: cần `configs/remap_<dataset>.yaml` ánh xạ tên class gốc → `banana` / `apple` / `orange` / `null` (bỏ box).
- **own/**: YOLO, tên class trong `own/data.yaml` là 6 class (`banana_fresh`…) hoặc 3 class (`banana`…) → tự đổi, không cần remap.

**Ô sẽ dừng** (không ghi gì) nếu thiếu nhãn/remap, có class chưa ánh xạ, đích ánh xạ sai, XML lỗi, hoặc class id ngoài `names`. Notebook **không tự đoán** ánh xạ.

In [ ]:
RF_SUFFIX = re.compile(r"(_(jpg|jpeg|png|bmp|webp))?\.rf\.[0-9A-Za-z]+$", re.I)


def own_source(stem):
    """'banana_vid1_000120_jpg.rf.ab12' -> 'own-banana_vid1' (bỏ hậu tố Roboflow và số thứ tự cuối)."""
    s = RF_SUFFIX.sub("", stem)
    m = re.match(r"^(.*)[_-]\d+$", s)
    return OWN_PREFIX + safe_name(m.group(1) if m and m.group(1) else s)


# --- Danh sách nguồn dữ liệu ---
jobs = [dict(name=d.name, root=d, origin="public", remap=CONFIGS / f"remap_{d.name}.yaml")
        for d in sorted(RAW.iterdir()) if d.is_dir()] if RAW.exists() else []
if list_images(OWN):
    jobs.append(dict(name="own", root=OWN, origin="own", remap=None))
assert jobs, f"Không tìm thấy dataset nào trong {RAW} hoặc {OWN}"

# --- Lượt 1: KIỂM TRA + chuyển đổi trong bộ nhớ (chưa ghi gì) ---
problems = []
for job in jobs:
    tag, root = job["name"], job["root"]
    imgs = list_images(root)
    dy = find_data_yaml(root)
    xmls = {p.stem: p for p in root.rglob("*.xml")}
    fmt = "yolo" if dy is not None else ("voc" if xmls else None)
    if fmt is None:
        problems.append(f"[{tag}] không có data.yaml (YOLO) cũng không có .xml (VOC): không đọc được nhãn")
        continue
    if job["origin"] == "own" and fmt != "yolo":
        problems.append(f"[{tag}] own/ phải ở định dạng YOLO có data.yaml")
        continue

    # Đọc nhãn thô: YOLO -> [(id_gốc hoặc None, các token còn lại)], VOC -> [(tên, x, y, w, h)]
    raw_labels, names = {}, None
    if fmt == "yolo":
        names = load_names(dy)
        bad_ids = set()
        for p in imgs:
            lp = label_path_for(p)
            if not lp.exists():
                raw_labels[p] = None
                continue
            rows = []
            for line in lp.read_text(errors="replace").splitlines():
                tok = line.split()
                if not tok:
                    continue
                try:
                    c = int(float(tok[0]))
                except ValueError:
                    rows.append((None, line.strip()))        # dòng hỏng: giữ nguyên để audit 1.3 báo
                    continue
                if not 0 <= c < len(names):
                    bad_ids.add(c)
                    continue
                rows.append((names[c], " ".join(tok[1:])))
            raw_labels[p] = rows
        if bad_ids:
            problems.append(f"[{tag}] nhãn dùng class id {sorted(bad_ids)} không có trong names={names}")
    else:
        found = set()
        for p in imgs:
            xp = xmls.get(p.stem)
            if xp is None:
                raw_labels[p] = None
                continue
            try:
                w, h, objs = read_voc(xp)
            except Exception as e:
                problems.append(f"[{tag}] XML lỗi {xp.name}: {e}")
                continue
            if not (w and h):
                im = cv2.imread(str(p))
                if im is None:
                    problems.append(f"[{tag}] không đọc được ảnh {p.name} để lấy kích thước")
                    continue
                h, w = im.shape[:2]
            rows = []
            for n, x1, y1, x2, y2 in objs:
                found.add(n)
                x1, x2 = max(0.0, min(x1, x2)), min(float(w), max(x1, x2))
                y1, y2 = max(0.0, min(y1, y2)), min(float(h), max(y1, y2))
                rows.append((n, f"{(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}"))
            raw_labels[p] = rows
        names = sorted(found)

    # Ánh xạ tên gốc -> loại quả
    if job["origin"] == "own":
        unknown = [n for n in names if n not in FRUITS and n not in CLASS_NAMES]
        if unknown:
            problems.append(f"[own] tên class trong own/data.yaml phải là {FRUITS} hoặc {CLASS_NAMES}; lạ: {unknown}")
            continue
        mapping = {n: (n if n in FRUITS else split_class(n)[0]) for n in names}
        job["has_state"] = all(n in CLASS_NAMES for n in names)
    elif job["remap"].exists():
        mapping = load_remap(job["remap"], names, FRUITS, problems, tag)
        job["has_state"] = False
    else:
        problems.append(f"[{tag}] thiếu file remap {job['remap']}; class gốc: {names}")
        continue
    if mapping is None:
        continue
    job.update(fmt=fmt, names=names, mapping=mapping, imgs=imgs, raw_labels=raw_labels)
    print(f"[{tag}] {fmt.upper()}, {len(imgs)} ảnh, class gốc {names}")

if problems:
    print("\n=== DỪNG: cần sửa trước khi gộp ===")
    print("\n".join(problems))
    raise RuntimeError("Có dataset thiếu nhãn/remap hoặc class chưa ánh xạ (xem danh sách ở trên). Không ghi gì vào work/merged_det.")

# --- Lượt 2: GHI vào work/merged_det ---
if ensure_fresh_dir(MERGED, "merged"):
    (MERGED / "images").mkdir()
    (MERGED / "labels").mkdir()
    manifest, tasks, seen = [], [], {}
    for job in jobs:
        tag = job["name"]
        stat = dict(images=0, boxes_in=0, boxes_out=0, dropped=0, no_label=0)
        for p in job["imgs"]:
            if p not in job["raw_labels"]:
                continue
            src = own_source(p.stem) if job["origin"] == "own" else safe_name(tag)
            new_name = f"{src}{SRC_SEP}{p.name}"
            if new_name in seen:
                raise RuntimeError(f"Trùng tên sau khi gộp: {p} và {seen[new_name]}")
            seen[new_name] = p
            rows = job["raw_labels"][p]
            out_lines = None
            if rows is None:
                stat["no_label"] += 1
            else:
                out_lines = []
                for n, rest in rows:
                    stat["boxes_in"] += 1
                    if n is None:
                        out_lines.append(rest)                       # dòng hỏng, để audit báo
                        continue
                    fruit = job["mapping"][n]
                    if fruit is None:
                        stat["dropped"] += 1                         # chủ ý bỏ theo remap (null)
                        continue
                    out_lines.append(f"{FRUIT_TO_ID[fruit]} {rest}")
                    stat["boxes_out"] += 1
            stat["images"] += 1
            tasks.append((p, new_name, out_lines))
            manifest.append(dict(merged_name=new_name, source=src, origin=job["origin"], dataset=tag,
                                 fmt=job["fmt"], has_state=bool(job["has_state"]), orig_image=str(p)))
        print(f"[{tag}] {stat}")

    def _write(t):
        p, new_name, out_lines = t
        shutil.copyfile(p, MERGED / "images" / new_name)
        if out_lines is not None:   # ảnh không có file nhãn -> không tạo nhãn (audit sẽ báo)
            (MERGED / "labels" / (Path(new_name).stem + ".txt")).write_text("\n".join(out_lines) + ("\n" if out_lines else ""))

    parallel_map(_write, tasks)
    pd.DataFrame(manifest).to_csv(MERGED / "manifest.csv", index=False)
    print(f"Đã gộp {len(tasks)} ảnh vào {MERGED}")
    print("Các nguồn:", sorted({m['source'] for m in manifest}))

### 1.3 Kiểm tra chất lượng (audit) dữ liệu detection
Kiểm tra trên `work/merged_det/`: ảnh có file nhãn tương ứng, nhãn đúng định dạng YOLO (`class x y w h`, toạ độ 0-1, id 0-2), ảnh đọc được.
In số ảnh, số box **mỗi class**, cảnh báo class dưới ngưỡng, tỷ lệ box lỗi, và vẽ 30 ảnh ngẫu nhiên để bạn **kiểm tra bằng mắt**
(đặc biệt với bộ VOC vừa đổi sang YOLO). Notebook chỉ báo cáo, không tự sửa nhãn.

In [ ]:
imgs = list_images(MERGED / "images")
assert imgs, f"{MERGED}/images trống: chạy ô 1.2 trước"
lbl_dir = MERGED / "labels"

def _audit_one(p):
    lp = lbl_dir / (p.stem + ".txt")
    boxes, bad = read_yolo_label(lp, nc=ND)
    readable = cv2.imread(str(p), cv2.IMREAD_REDUCED_GRAYSCALE_4) is not None   # chỉ để biết ảnh giải mã được không
    return p, lp.exists(), boxes, bad, readable

results = parallel_map(_audit_one, imgs)
cls_count = np.zeros(ND, int)
missing_label, unreadable, bad_rows = [], [], []
n_box_ok = 0
for p, has_lbl, boxes, bad, readable in results:
    if not has_lbl:
        missing_label.append(p.name)
    if not readable:
        unreadable.append(p.name)
    for b in boxes:
        cls_count[b[0]] += 1
    n_box_ok += len(boxes)
    bad_rows += [dict(file=p.stem + ".txt", line=ln, content=txt, reason=r) for ln, txt, r in bad]
img_stems = {p.stem for p in imgs}
orphan_labels = [lp.name for lp in lbl_dir.glob("*.txt") if lp.stem not in img_stems]

n_lines = n_box_ok + len(bad_rows)
print(f"Số ảnh: {len(imgs)} | ảnh thiếu file nhãn: {len(missing_label)} | ảnh không đọc được: {len(unreadable)} | nhãn mồ côi: {len(orphan_labels)}")
print(f"Số dòng nhãn: {n_lines} | box hợp lệ: {n_box_ok} | box lỗi: {len(bad_rows)} "
      f"({(100 * len(bad_rows) / n_lines) if n_lines else 0:.2f}%)")

cls_df = pd.DataFrame(dict(class_id=range(ND), class_name=FRUITS, n_box=cls_count))
cls_df["canh_bao"] = ["" if n >= MIN_BOX_TRAIN else f"ÍT: {n} < {MIN_BOX_TRAIN} (ngưỡng train)" for n in cls_count]
display(cls_df)
for r in cls_df.itertuples():
    if r.canh_bao:
        print(f"⚠️  {r.class_name}: {r.canh_bao}")
if bad_rows:
    display(pd.DataFrame(bad_rows).head(20))
    print("Lý do lỗi:", pd.DataFrame(bad_rows).reason.value_counts().to_dict())

# Vẽ box lên N ảnh ngẫu nhiên (cố định theo SEED)
sample = random.Random(SEED).sample(imgs, min(N_AUDIT_SAMPLES, len(imgs)))
save_dir = AUDIT_DIR / "samples" if ensure_fresh_dir(AUDIT_DIR, "audit") else LOCAL_DATASET / "_audit_tmp"
save_dir.mkdir(parents=True, exist_ok=True)       # đã có audit cũ -> chỉ lưu tạm cục bộ để hiển thị
drawn = []
for p in sample:
    img = cv2.imread(str(p))
    if img is None:
        continue
    h, w = img.shape[:2]
    for c, *xyxy in yolo_to_xyxy(read_yolo_label(lbl_dir / (p.stem + ".txt"), nc=ND)[0], w, h):
        draw_box(img, xyxy, FRUITS[c], (0, 255, 0))
    cv2.imwrite(str(save_dir / p.name), img)
    drawn.append(save_dir / p.name)
if save_dir.is_relative_to(AUDIT_DIR):
    cls_df.to_csv(AUDIT_DIR / "class_counts.csv", index=False)
    pd.DataFrame(bad_rows, columns=["file", "line", "content", "reason"]).to_csv(AUDIT_DIR / "bad_labels.csv", index=False)
    pd.Series(missing_label, name="image").to_csv(AUDIT_DIR / "missing_labels.csv", index=False)
    pd.Series(unreadable, name="image").to_csv(AUDIT_DIR / "unreadable_images.csv", index=False)
    print("Đã lưu báo cáo audit vào", AUDIT_DIR)
show_grid(drawn, ncols=5, title="Kiểm tra bằng mắt: box và loại quả")

### 1.4 Chia tập detection theo NHÓM ảnh (không chia ngẫu nhiên theo ảnh)
Ảnh cùng nhóm rất giống nhau (frame liền nhau của video, ảnh augment, ảnh trùng lặp). Chia ngẫu nhiên theo ảnh sẽ làm test "giống" train và điểm bị ảo.
Vì vậy **mỗi nhóm được gán trọn vào một tập**:
- **Nguồn tự quay (`own-*`)**: cả nguồn (một video/buổi chụp) là một nhóm.
- **Ảnh public**: cụm ảnh gần giống (dHash 64-bit cách nhau <= `DUP_HASH_DIST` bit, so cả bản lật ngang) hoặc cùng ảnh gốc Roboflow (`<tên>_jpg.rf.<mã>`).
  Một cụm có thể gồm ảnh của nhiều dataset nếu các dataset đó chứa ảnh trùng nhau.

Thuật toán (cố định theo `SEED`): xáo nhóm → sắp giảm dần theo số box → gán từng nhóm vào tập **đang thiếu nhiều box nhất theo từng class** so với 70/15/15.
Nhóm public và nhóm own chia **riêng** để test chắc chắn có ảnh tự quay.

- Baseline detection train **chỉ dùng ảnh public** của tập train; ảnh `own-*` thuộc train để dành cho fine-tune. Val/test dùng mọi nguồn.
- Phép chia ảnh tự quay ở đây cũng quyết định crop tự quay thuộc train/val/test của classifier (1.6), nên hai tầng **không rò rỉ chéo**.
- Ghi `work/split_images.csv` và `work/split_report.csv`. Đã có và `OVERWRITE['split']=False` → **dùng lại** để tập test không đổi.
**Ảnh tự quay** (`OWN_SPLIT_MODE` ở CONFIG): `"block"` (mặc định) chia theo khối `OWN_BLOCK_FRAMES` frame liên tiếp, cân bằng theo 6 class, nên train/val/test đều có đủ loại quả.
Đổi lại test là *cùng vật thể ở thời điểm khác*, chưa phải vật thể mới. `"video"` giữ cả video trong một tập (test là video chưa thấy) nhưng với ít video thì test dễ thiếu loại quả.


In [ ]:
imgs = list_images(MERGED / "images")
assert imgs, "Chạy ô 1.2 trước"
names_all = [p.name for p in imgs]

def _count(p):
    c = np.zeros(ND, int)
    for b in read_yolo_label(MERGED / "labels" / (p.stem + ".txt"), nc=ND)[0]:
        c[b[0]] += 1
    return c

counts = np.array(parallel_map(_count, imgs)).reshape(-1, ND)
df_img = pd.DataFrame(counts, columns=FRUITS)
df_img.insert(0, "image", names_all)
df_img.insert(1, "source", [source_of(n) for n in names_all])
df_img.insert(2, "origin", ["own" if is_own_source(s) else "public" for s in df_img.source])
df_img["n_boxes"] = counts.sum(1)

if SPLIT_IMAGES.exists() and not OVERWRITE["split"]:
    old = pd.read_csv(SPLIT_IMAGES)
    if set(old.image) != set(names_all):
        raise RuntimeError(f"Tập ảnh trong work/merged_det khác {SPLIT_IMAGES.name} (thêm {len(set(names_all) - set(old.image))}, "
                           f"mất {len(set(old.image) - set(names_all))} ảnh). Đặt OVERWRITE['split']=True để chia lại "
                           "(tập test sẽ đổi, Phần 4 sẽ tự đánh giá lại baseline).")
    df_img = df_img.merge(old[["image", "group", "split"]], on="image")
    print(f"Dùng lại phép chia cũ trong {SPLIT_IMAGES}")
else:
    t0 = time.time()
    pub = [i for i in range(len(df_img)) if df_img.origin[i] == "public"]
    keys = [(df_img.source[i], rf_base(Path(names_all[i].split(SRC_SEP, 1)[1]).stem)) for i in pub]
    keys = [k if k[1] is not None else None for k in keys]
    roots = cluster_images([imgs[i] for i in pub], keys)
    group = list(df_img.source)                                    # mode "video": cả nguồn là một nhóm
    own_idx = [i for i in range(len(df_img)) if df_img.origin[i] == "own"]
    if OWN_SPLIT_MODE == "block":                                  # khối frame liên tiếp của cùng video là một nhóm
        for src_name, idxs in df_img.iloc[own_idx].groupby("source").groups.items():
            for rank, i in enumerate(sorted(idxs, key=lambda k: names_all[k])):
                group[i] = f"{src_name}#blk{rank // OWN_BLOCK_FRAMES:03d}"
    for j, i in enumerate(pub):
        group[i] = f"{df_img.source[pub[roots[j]]]}#{pub[roots[j]]:06d}"
    df_img["group"] = group
    print(f"Gom nhóm xong sau {time.time() - t0:.0f}s: {df_img.group.nunique()} nhóm cho {len(df_img)} ảnh")
    grp_df = df_img.groupby("group").agg(n_images=("image", "size"), n_boxes=("n_boxes", "sum"),
                                         origin=("origin", "first"), **{c: (c, "sum") for c in FRUITS})
    assignment = {}
    for origin, g in grp_df.groupby("origin"):
        cols = FRUITS
        if origin == "own":                                        # cân bằng theo 6 class (loại quả + tình trạng) nếu own/ có nhãn tình trạng
            man = pd.read_csv(MERGED / "manifest.csv").set_index("merged_name")
            own_names = load_names(find_data_yaml(OWN))
            cols = CLASS_NAMES if all(n in CLASS_NAMES for n in own_names) else FRUITS
            rows = []
            for i in own_idx:
                c = dict.fromkeys(cols, 0)
                for b in read_yolo_label(label_path_for(Path(man.orig_image[names_all[i]])), nc=len(own_names))[0]:
                    n = own_names[b[0]]
                    c[n if n in cols else split_class(n)[0]] += 1
                rows.append(c)
            g = pd.DataFrame(rows).groupby(df_img.group.iloc[own_idx].to_numpy()).sum()
            g["n_boxes"] = g[cols].sum(1)
        warn_big_groups(g, "n_boxes", f"detection/{origin}")
        assignment.update(assign_groups(g, SPLIT_RATIOS, SEED, cols, "n_boxes"))
    df_img["split"] = df_img.group.map(assignment)
    write_text_guarded(SPLIT_IMAGES, df_img.to_csv(index=False), "split")
    rep_grp = df_img.groupby(["split", "origin", "source", "group"]).agg(
        n_images=("image", "size"), n_boxes=("n_boxes", "sum"), **{c: (c, "sum") for c in FRUITS}).reset_index()
    write_text_guarded(SPLIT_REPORT, rep_grp.to_csv(index=False), "split")
    print(f"Đã ghi {SPLIT_IMAGES.name} và {SPLIT_REPORT.name} trong {WORK}")

display(df_img.groupby(["split", "origin", "source"]).agg(n_groups=("group", "nunique"), n_images=("image", "size"),
                                                         n_boxes=("n_boxes", "sum")).reset_index())
tot = df_img.groupby("split").n_boxes.sum()
print("Tỷ lệ box thực tế:", {k: f"{100 * v / max(1, tot.sum()):.1f}%" for k, v in tot.items()})
for origin, g in df_img.groupby("origin"):
    t = g.groupby("split").n_boxes.sum()
    print(f"  nhóm {origin}:", {k: f"{100 * v / max(1, t.sum()):.1f}%" for k, v in t.items()}, f"({g.group.nunique()} nhóm)")

**Kiểm tra chống rò rỉ** và cảnh báo thiếu class ở val, test. Ô này dừng (`assert`) nếu một nhóm (hoặc nguồn tự quay) nằm ở hai tập.

In [ ]:
df_img = pd.read_csv(SPLIT_IMAGES)
assert df_img.image.is_unique, "Một ảnh xuất hiện nhiều lần"
assert set(df_img.split) <= set(SPLIT_RATIOS), f"Tên tập lạ: {set(df_img.split)}"
n_split = df_img.groupby("group").split.nunique()
assert (n_split == 1).all(), f"RÒ RỈ: nhóm nằm ở nhiều tập: {list(n_split[n_split > 1].index[:10])}"
if OWN_SPLIT_MODE == "video":
    n_split_own = df_img[df_img.origin == "own"].groupby("source").split.nunique()
    assert (n_split_own == 1).all(), f"RÒ RỈ: nguồn tự quay nằm ở nhiều tập: {list(n_split_own[n_split_own > 1].index)}"
assert (df_img.split == "val").any() and (df_img.split == "test").any(), "Val hoặc test rỗng: cần thêm dữ liệu"
print("✅ Không nhóm ảnh / nguồn nào nằm ở hai tập.")

df_img["subset"] = np.where(df_img.split == "train", "train_" + df_img.origin, df_img.split)
per = df_img.groupby("subset")[FRUITS].sum().T
display(per)
for sub in per.columns:
    if sub == "train_own":
        continue                                     # dữ liệu fine-tune: số lượng tuỳ người dùng
    thr = MIN_BOX_TRAIN if sub.startswith("train") else MIN_BOX_EVAL
    for c in FRUITS:
        n = int(per.loc[c, sub])
        if n == 0:
            print(f"⚠️  {sub}: THIẾU class {c}")
        elif n < thr:
            print(f"⚠️  {sub}: {c} chỉ có {n} box (< {thr})")
if not (df_img.origin == "own").any():
    print("⚠️  Chưa có ảnh tự quay: Phần 4 (fine-tune) và đánh giá end-to-end sẽ không chạy được.")

In [ ]:
# Sau khi chia lại có ảnh tự quay: ảnh public phải GIỮ NGUYÊN tập, nếu không det_baseline (đã train) có thể bị rò rỉ sang test mới.
prev = WORK / "split_images.prev.csv"
if prev.exists():
    old = pd.read_csv(prev)
    if "origin" in old:
        old = old[old.origin == "public"]
    new = pd.read_csv(SPLIT_IMAGES)[["image", "split"]]
    m = old[["image", "split"]].merge(new, on="image", how="left", suffixes=("_cu", "_moi"))
    n_chg = int((m.split_cu != m.split_moi).sum())
    if n_chg:
        print(f"⚠️  {n_chg} ảnh public đổi tập so với phép chia cũ. det_baseline/cls đã train trên phép chia cũ có thể "
              "thấy ảnh nay thuộc val/test: đặt OVERWRITE['det_baseline']=True (và OVERWRITE['cls_baseline']=True) rồi train lại.")
    else:
        print(f"✅ {len(m)} ảnh public giữ nguyên tập: baseline đã train vẫn hợp lệ.")
if AUTO_FLAGGED:                       # tắt cờ ghi đè do ô 1.1b bật, tránh ghi đè ở các ô sau (cls_split, data_yaml)
    for k in ("merged", "audit", "split"):
        OVERWRITE[k] = False
    AUTO_FLAGGED = False

### 1.5 Tạo `work/data.yaml` và chép dữ liệu detection sang ổ cục bộ Colab
`data.yaml` dùng đường dẫn tuyệt đối trong `/content/dataset/det`. Đọc ảnh từ ổ cục bộ nhanh hơn nhiều so với từ Drive.

> `/content/dataset` **mất khi phiên Colab kết thúc**: mỗi phiên mới cần chạy lại Phần 0, ô chép này và ô 1.6b (dữ liệu gốc vẫn trên Drive).

In [ ]:
data_cfg = {
    "path": str(LOCAL_DET),
    "train": str(LOCAL_DET / "train" / "images"),
    "val": str(LOCAL_DET / "val" / "images"),
    "test": str(LOCAL_DET / "test" / "images"),
    "test_public": str(LOCAL_DET / "test_public" / "images"),      # tách test để so với baseline cũ (public-only) và đo riêng ảnh tự quay
    "test_own": str(LOCAL_DET / "test_own" / "images"),
    "nc": ND,
    "names": {i: n for i, n in enumerate(FRUITS)},
}
write_text_guarded(DATA_YAML, yaml.safe_dump(data_cfg, sort_keys=False, allow_unicode=True), "split")
print(DATA_YAML.read_text())

In [ ]:
def build_local_split(split_name, image_names):
    """Chép ảnh + nhãn từ work/merged_det (Drive) sang LOCAL_DET/<split>. Bỏ qua nếu thư mục đã đúng danh sách ảnh."""
    dst = LOCAL_DET / split_name
    fp = fingerprint(image_names)
    if (dst / ".fp").exists() and (dst / ".fp").read_text() == fp:
        print(f"det/{split_name:9s}: {len(image_names):6d} ảnh (đã có, bỏ qua chép)")
        return
    out = fresh_local_dir(dst)
    (out / "images").mkdir()
    (out / "labels").mkdir()
    def _copy(name):
        shutil.copyfile(MERGED / "images" / name, out / "images" / name)
        lp = MERGED / "labels" / (Path(name).stem + ".txt")
        if lp.exists():
            shutil.copyfile(lp, out / "labels" / lp.name)
    parallel_map(_copy, image_names)
    (out / ".fp").write_text(fp)
    print(f"det/{split_name:9s}: {len(image_names):6d} ảnh -> {out}")


si = pd.read_csv(SPLIT_IMAGES)
t0 = time.time()
for sp in ("train", "val", "test"):
    sel = si[si.split == sp]
    if sp == "train":
        sel = sel[sel.origin == "public"]     # ảnh tự quay thuộc train: để dành cho fine-tune (Phần 4)
    build_local_split(sp, sorted(sel.image))
for origin in ("public", "own"):             # test tách theo nguồn (chỉ để báo cáo; tập test chính vẫn là "test")
    sel = si[(si.split == "test") & (si.origin == origin)]
    if len(sel):
        build_local_split(f"test_{origin}", sorted(sel.image))
print(f"Xong sau {time.time() - t0:.0f}s")

### 1.6 Dữ liệu classifier fresh/rotten (tầng 2)

**1.6a Bộ phân loại public** (`raw_cls/<dataset>/`, ví dụ các bộ fresh/rotten trên Kaggle): ảnh nằm trong thư mục theo class, không cần box.
- Cần `configs/remap_cls_<dataset>.yaml` ánh xạ **tên thư mục chứa ảnh** → một trong 6 class (`banana_fresh` … `orange_rotten`) hoặc `null` (bỏ).
  Thư mục cùng tên ở `train/` và `test/` của dataset gốc được coi là cùng class; notebook tự chia lại.
- Chia train/val/test **theo nhóm ảnh gần giống** (dHash, như 1.4), cân theo từng (loại quả, tình trạng), ghi `work/split_cls.csv`.
  Tập test public chỉ để tham khảo: **test chính của classifier là crop tự quay** (1.6b).

In [ ]:
problems, recs = [], []
ds_dirs = sorted(d for d in RAW_CLS.iterdir() if d.is_dir()) if RAW_CLS.exists() else []
assert ds_dirs, f"Chưa có dataset phân loại trong {RAW_CLS}: classifier cần ít nhất một bộ fresh/rotten"
for d in ds_dirs:
    tag = f"cls/{d.name}"
    imgs = list_images(d)
    leaves = sorted({p.parent.name for p in imgs})
    rf = CONFIGS / f"remap_cls_{d.name}.yaml"
    if not rf.exists():
        problems.append(f"[{tag}] thiếu {rf}; các thư mục chứa ảnh: {leaves}")
        continue
    mp = load_remap(rf, leaves, CLASS_NAMES, problems, tag)
    if mp is None:
        continue
    n_drop = 0
    for p in imgs:
        t = mp[p.parent.name]
        if t is None:
            n_drop += 1
            continue
        f, s = split_class(t)
        recs.append(dict(path=str(p.relative_to(RAW_CLS)), source=safe_name(d.name), fruit=f, state=s, cls6=t))
    print(f"[{tag}] {len(imgs)} ảnh, thư mục {leaves}, bỏ {n_drop} ảnh theo remap (null)")
if problems:
    print("\n=== DỪNG: cần sửa trước khi tạo dữ liệu classifier ===")
    print("\n".join(problems))
    raise RuntimeError("Có dataset phân loại thiếu remap / thư mục chưa ánh xạ (xem danh sách ở trên).")

df_cls = pd.DataFrame(recs)
assert len(df_cls), "Không còn ảnh nào sau remap"
if SPLIT_CLS.exists() and not OVERWRITE["split"]:
    old = pd.read_csv(SPLIT_CLS)
    if set(old.path) != set(df_cls.path):
        raise RuntimeError(f"Tập ảnh phân loại khác {SPLIT_CLS.name}. Đặt OVERWRITE['split']=True để chia lại (test public sẽ đổi).")
    df_cls = df_cls.merge(old[["path", "group", "split"]], on="path")
    print(f"Dùng lại phép chia cũ trong {SPLIT_CLS}")
else:
    t0 = time.time()
    roots = cluster_images([RAW_CLS / p for p in df_cls.path])
    df_cls["group"] = [f"{df_cls.source[r]}#{r:06d}" for r in roots]
    onehot = pd.get_dummies(df_cls.cls6).reindex(columns=CLASS_NAMES, fill_value=0).astype(int)
    g = pd.concat([df_cls[["group"]], onehot], axis=1).groupby("group").sum()
    g["n_images"] = g[CLASS_NAMES].sum(1)
    print(f"Gom nhóm xong sau {time.time() - t0:.0f}s: {len(g)} nhóm cho {len(df_cls)} ảnh")
    warn_big_groups(g, "n_images", "classifier/public")
    asg = assign_groups(g, SPLIT_RATIOS, SEED, CLASS_NAMES, "n_images")
    df_cls["split"] = df_cls.group.map(asg)
    write_text_guarded(SPLIT_CLS, df_cls.to_csv(index=False), "split")
    print(f"Đã ghi {SPLIT_CLS}")

# Kiểm tra chống rò rỉ + thống kê
n_split = df_cls.groupby("group").split.nunique()
assert (n_split == 1).all(), f"RÒ RỈ: nhóm ảnh phân loại nằm ở nhiều tập: {list(n_split[n_split > 1].index[:10])}"
print("✅ Không nhóm ảnh phân loại nào nằm ở hai tập.")
tab = pd.crosstab(df_cls.cls6, df_cls.split).reindex(CLASS_NAMES, fill_value=0)
display(tab)
for c in CLASS_NAMES:
    if "train" not in tab.columns or tab.loc[c, "train"] == 0:
        print(f"⚠️  train public THIẾU {c}: classifier sẽ không học được tổ hợp này từ dữ liệu public")
    elif tab.loc[c, "train"] < MIN_CROP_TRAIN:
        print(f"⚠️  train public: {c} chỉ có {tab.loc[c, 'train']} ảnh (< {MIN_CROP_TRAIN})")

**1.6b Dựng dữ liệu classifier trên ổ cục bộ** (`/content/dataset/cls`):

| Thư mục | Nội dung | Dùng cho |
|---|---|---|
| `base/train`, `base/val` | ảnh public | train classifier baseline |
| `own/train`, `own/val` | crop từ ảnh tự quay thuộc train / val (theo phép chia 1.4) | fine-tune (Phần 4) |
| `eval/test_own` | crop từ ảnh tự quay thuộc **test** | **test chính** của classifier |
| `eval/test_public` | ảnh public thuộc test | test tham khảo (cùng miền với dữ liệu train) |

Crop lấy từ nhãn 6 class trong `own/`: cắt theo box, nới thêm `CROP_PAD`, bỏ crop nhỏ hơn `CROP_MIN_PX`. Mỗi thư mục có `fresh/` và `rotten/`.
Tên crop `<nguồn>__<loại_quả>__<tên>.jpg` để đánh giá được theo từng loại quả.

In [ ]:
SETS = {"base/train": None, "base/val": None, "own/train": None, "own/val": None, "eval/test_own": None, "eval/test_public": None}
for top in ("base", "own", "eval"):
    fresh_local_dir(LOCAL_CLS / top)
for s in SETS:
    for st in STATES:
        (LOCAL_CLS / s / st).mkdir(parents=True, exist_ok=True)

# --- Ảnh public: chép theo split_cls.csv ---
dfc = pd.read_csv(SPLIT_CLS)
pub_dest = {"train": "base/train", "val": "base/val", "test": "eval/test_public"}

def _pub(r):
    h8 = hashlib.md5(r.path.encode()).hexdigest()[:8]       # tránh trùng tên giữa các thư mục con
    name = f"{r.source}{SRC_SEP}{r.fruit}{SRC_SEP}{safe_name(Path(r.path).stem)}-{h8}{Path(r.path).suffix.lower()}"
    shutil.copyfile(RAW_CLS / r.path, LOCAL_CLS / pub_dest[r.split] / r.state / name)

t0 = time.time()
parallel_map(_pub, list(dfc.itertuples()))

# --- Cân bằng base/train: mỗi loại quả, tình trạng nhiều hơn bị giảm còn <= CLS_BALANCE_RATIO x tình trạng ít hơn ---
if CLS_BALANCE_RATIO:
    rng = random.Random(SEED)
    for f in FRUITS:
        per = {st: sorted(p for p in list_images(LOCAL_CLS / "base/train" / st) if fruit_of_crop(p.name) == f) for st in STATES}
        lo = min(len(v) for v in per.values())
        for st, v in per.items():
            cap = int(CLS_BALANCE_RATIO * lo)
            if lo and len(v) > cap:
                for p in rng.sample(v, len(v) - cap):
                    p.unlink()
                print(f"cân bằng base/train/{f}: {st} {len(v)} -> {cap}")

# --- Crop từ ảnh tự quay có nhãn 6 class ---
man = pd.read_csv(MERGED / "manifest.csv")
si = pd.read_csv(SPLIT_IMAGES)
own = man[(man.origin == "own") & (man.has_state.astype(str) == "True")].merge(
    si[["image", "split"]], left_on="merged_name", right_on="image")
own_dest = {"train": "own/train", "val": "own/val", "test": "eval/test_own"}
crop_stats = []
if len(own):
    own_names = load_names(find_data_yaml(OWN))

    def _crop(r):
        img = cv2.imread(r.orig_image)
        if img is None:
            return [("unreadable", None, None)]
        h, w = img.shape[:2]
        boxes, _ = read_yolo_label(label_path_for(Path(r.orig_image)), nc=len(own_names))
        out = []
        stem = Path(r.merged_name.split(SRC_SEP, 1)[1]).stem
        for k, (c, *xyxy) in enumerate(yolo_to_xyxy(boxes, w, h)):
            f, s = split_class(own_names[c])
            if min(xyxy[2] - xyxy[0], xyxy[3] - xyxy[1]) < CROP_MIN_PX:
                out.append(("too_small", f, s))
                continue
            name = f"{r.source}{SRC_SEP}{f}{SRC_SEP}{safe_name(stem)}_{k}.jpg"
            cv2.imwrite(str(LOCAL_CLS / own_dest[r.split] / s / name), crop_box(img, xyxy))
            out.append((own_dest[r.split], f, s))
        return out

    for res in parallel_map(_crop, list(own.itertuples())):
        crop_stats += res
    skipped = pd.Series([a for a, _, _ in crop_stats if a in ("too_small", "unreadable")]).value_counts().to_dict()
    print(f"Crop tự quay: {sum(a in own_dest.values() for a, _, _ in crop_stats)} crop, bỏ: {skipped}")
else:
    print("⚠️  Chưa có ảnh tự quay gán nhãn 6 class: không có test classifier tự quay (eval/test_own) và không fine-tune được tầng 2.")
print(f"Xong sau {time.time() - t0:.0f}s")


def cls_counts():
    """Đếm số ảnh mỗi (tập, loại quả, tình trạng) trên /content/dataset/cls."""
    rows = []
    for s in SETS:
        for st in STATES:
            for p in list_images(LOCAL_CLS / s / st):
                rows.append((s, f"{fruit_of_crop(p.name)}_{st}"))
    df = pd.DataFrame(rows, columns=["set", "cls6"])
    return pd.crosstab(df.cls6, df.set).reindex(index=CLASS_NAMES, columns=list(SETS), fill_value=0)


tab = cls_counts()
display(tab)
for s, thr in (("base/train", MIN_CROP_TRAIN), ("eval/test_own", MIN_CROP_EVAL)):
    for c in CLASS_NAMES:
        n = int(tab.loc[c, s])
        if n < thr:
            print(f"⚠️  {s}: {c} có {n} ảnh (< {thr})" + (": số liệu không ổn định" if s.startswith("eval") else ""))

# --- Kiểm tra: không crop nào ở hai tập, train/val mỗi tình trạng đều có ảnh ---
seen = {}
for s in SETS:
    for p in list_images(LOCAL_CLS / s):
        assert p.name not in seen, f"RÒ RỈ: {p.name} nằm ở cả {seen[p.name]} và {s}"
        seen[p.name] = s
for s in ("base/train", "base/val"):
    for st in STATES:
        assert list_images(LOCAL_CLS / s / st), f"{s}/{st} rỗng: classifier cần đủ fresh và rotten ở train/val"
print("✅ Không crop/ảnh nào nằm ở hai tập.")

## Phần 2. Train baseline
Hai tầng train **độc lập**. Kết quả ghi **thẳng vào Drive** (`runs/det_baseline`, `runs/cls_baseline`) nên không mất khi phiên bị ngắt;
`last.pt` lưu mỗi epoch. Augmentation HSV (độ sáng/màu) bật theo CONFIG.

Các ô train là ô **nặng**. Nếu thư mục run đã tồn tại, ô sẽ bỏ qua: dùng ô tiếp tục, hoặc đặt `OVERWRITE[...]=True` để train lại từ đầu.

### 2.0 Hàm train dùng chung (ô nhẹ, chạy trước mọi ô train/resume)

In [ ]:
from ultralytics import YOLO


def run_train(run, init_weights, **kw):
    """Train vào RUNS/<run>, bỏ qua nếu đã có (trừ khi OVERWRITE[run]) hoặc đang ở chế độ RESUME."""
    out = RUNS / run
    if RESUME[run]:
        print(f"RESUME['{run}']=True -> bỏ qua ô này, chạy ô tiếp tục bên dưới.")
        return
    if out.exists() and not OVERWRITE[run]:
        print(f"[BỎ QUA] {out} đã tồn tại. Train dở -> RESUME['{run}']=True và chạy ô tiếp tục; train lại -> OVERWRITE['{run}']=True.")
        return
    if out.exists():
        shutil.rmtree(out)
    YOLO(str(init_weights)).train(seed=SEED, deterministic=True, patience=PATIENCE,
                                  hsv_h=HSV_H, hsv_s=HSV_S, hsv_v=HSV_V,      # augmentation độ sáng / màu
                                  project=str(RUNS), name=run, exist_ok=True, plots=True, **kw)


def run_resume(run, data_check):
    """Tiếp tục từ RUNS/<run>/weights/last.pt nếu RESUME[run]=True."""
    if not RESUME[run]:
        print(f"RESUME['{run}']=False -> không làm gì.")
        return
    last = RUNS / run / "weights" / "last.pt"
    assert last.exists(), f"Không thấy {last}"
    assert Path(data_check).exists(), f"Chưa có {data_check}: chạy lại các ô chép dữ liệu (1.5, 1.6b, và 4.1 nếu fine-tune)"
    YOLO(str(last)).train(resume=True)


print("Đã định nghĩa run_train() và run_resume().")

### 2.1 Tầng 1: detection 3 class (baseline, chỉ ảnh public)

In [ ]:
run_train("det_baseline", f"{MODEL_SIZE}.pt", data=str(DATA_YAML), epochs=EPOCHS_BASE, imgsz=IMGSZ, batch=BATCH,
          optimizer=OPTIMIZER, lr0=LR0_BASE)

### 2.2 Tầng 2: classifier fresh/rotten (baseline, chỉ ảnh public)
Ultralytics đọc `base/train/<fresh|rotten>/` và `base/val/...`; thứ tự class theo tên thư mục (`fresh`=0, `rotten`=1).
Có thể chạy **song song** với 2.1 ở một phiên Colab khác (ở phiên đó chạy Phần 0, 1.6b và 2.0, rồi ô này).

In [ ]:
run_train("cls_baseline", f"{CLS_MODEL}.pt", data=str(LOCAL_CLS / "base"), epochs=EPOCHS_CLS, imgsz=CLS_IMGSZ,
          batch=CLS_BATCH, optimizer=CLS_OPTIMIZER, lr0=LR0_CLS)

### 2.3 Tiếp tục khi bị ngắt (CHỈ chạy khi cần)
Phiên mới → Phần 0 → ô 1.5 (chép detection) và 1.6b (dữ liệu classifier) → ô 2.0 →
đặt `RESUME['det_baseline']=True` hoặc `RESUME['cls_baseline']=True` trong CONFIG, chạy lại ô CONFIG → chạy ô này. Mặc định ô không làm gì.

In [ ]:
run_resume("det_baseline", LOCAL_DET / "train")
run_resume("cls_baseline", LOCAL_CLS / "base" / "train")

### 2.4 Đường cong huấn luyện

In [ ]:
hist_det_base = show_training(RUNS / "det_baseline")
hist_cls_base = show_training(RUNS / "cls_baseline")

### 2.5 Chép ngược kết quả vào Drive
Train đã ghi thẳng vào `runs/` trên Drive. Ô này kiểm tra weights có trên Drive và, nếu Ultralytics có ghi gì vào thư mục cục bộ mặc định
`/content/runs`, chép bản đó sang `runs/_local_backup/`.

In [ ]:
for run in ("det_baseline", "cls_baseline", "det_finetune", "det_joint", "cls_finetune"):
    for w in ("best.pt", "last.pt"):
        p = RUNS / run / "weights" / w
        print(f"{'[có]     ' if p.exists() else '[chưa có]'} {p}")
local_runs = Path("/content/runs")
if local_runs.exists() and any(local_runs.iterdir()):
    dst = RUNS / "_local_backup"
    shutil.copytree(local_runs, dst, dirs_exist_ok=True)
    print("Đã chép", local_runs, "->", dst)

## Phần 3. Đánh giá
Ba hàm, mỗi hàm dùng chung cho baseline và fine-tune; mọi bảng lưu vào `runs/<tag>/metrics/` trên Drive:

| Hàm | Đánh giá | Tập |
|---|---|---|
| `evaluate_det(weights, split, tag)` | tầng 1: P, R, mAP@0.5, mAP@0.5:0.95 theo loại quả, confusion matrix, ảnh lỗi, tốc độ | `det/test` (public + tự quay) |
| `evaluate_cls(weights, tag)` | tầng 2: accuracy, P/R/F1 theo tình trạng và theo loại quả, confusion matrix, crop lỗi, tốc độ | `test_own` (chính), `test_public` |
| `evaluate_e2e(det_w, cls_w, tag)` | cả pipeline: P/R/F1 theo 6 class, nhầm loại quả / nhầm tình trạng, **sai số đếm**, ảnh lỗi, tốc độ | ảnh tự quay thuộc test |

Số liệu của class ít mẫu **không ổn định**: bảng luôn hiện số mẫu và cột `canh_bao` để bạn báo cáo trung thực.

### 3.1 Hàm dùng chung cho đánh giá

In [ ]:
from ultralytics import YOLO
import torch
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix


def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def box_iou(a, b):
    """IoU giữa mảng box a (N,4) và b (M,4) dạng xyxy."""
    a, b = np.asarray(a, float).reshape(-1, 4), np.asarray(b, float).reshape(-1, 4)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(-1)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def match_boxes(gt_xyxy, pred_xyxy):
    """Ghép tham lam theo IoU giảm dần (IoU >= IOU_MATCH, không xét class). Trả về [(i_gt, j_pred)]."""
    if not len(gt_xyxy) or not len(pred_xyxy):
        return []
    iou = box_iou(gt_xyxy, pred_xyxy)
    pairs = sorted(((iou[i, j], i, j) for i in range(iou.shape[0]) for j in range(iou.shape[1]) if iou[i, j] >= IOU_MATCH),
                   reverse=True)
    used_g, used_p, out = set(), set(), []
    for _, i, j in pairs:
        if i not in used_g and j not in used_p:
            used_g.add(i); used_p.add(j); out.append((i, j))
    return out


def plot_confusion(M, row_labels, col_labels, title, path, ylabel, xlabel):
    fig, ax = plt.subplots(figsize=(1.1 * len(col_labels) + 2.5, 1.0 * len(row_labels) + 2))
    ax.imshow(M, cmap="Blues")
    ax.set_xticks(range(len(col_labels))); ax.set_xticklabels(col_labels, rotation=45, ha="right")
    ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if M[i, j]:
                ax.text(j, i, int(M[i, j]), ha="center", va="center", fontsize=8,
                        color="white" if M[i, j] > M.max() / 2 else "black")
    plt.tight_layout(); fig.savefig(path, dpi=150); plt.show()


def time_per_item(fn, items, warmup=5):
    """Thời gian trung bình (ms) mỗi lần gọi fn(item) trên GPU, sau vài lần chạy làm nóng."""
    items = list(items)
    for it in items[:warmup]:
        fn(it)
    cuda_sync()
    t0 = time.perf_counter()
    for it in items:
        fn(it)
    cuda_sync()
    return (time.perf_counter() - t0) * 1000 / max(1, len(items))


print("Đã định nghĩa hàm đánh giá dùng chung.")

### 3.2 `evaluate_det()`: tầng 1
1. `model.val(split=...)` → P, R, mAP@0.5, mAP@0.5:0.95 theo từng loại quả, kèm số box thật và cảnh báo ít mẫu.
2. Confusion matrix 3 loại quả + nền (hàng = dự đoán, cột = nhãn thật); đếm nhầm loại quả, bỏ sót, báo nhầm.
3. `N_ERROR_IMAGES` ảnh sai / độ tin cậy thấp, vẽ box thật (xanh lá) và dự đoán (đỏ).
4. Tốc độ suy luận trung bình (ms/ảnh, batch=1, gồm đọc ảnh và tiền/hậu xử lý).

In [ ]:
def evaluate_det(weights, split="test", tag="det_baseline"):
    weights = Path(weights)
    assert weights.exists(), f"Không thấy {weights}"
    assert (LOCAL_DET / split / "images").exists(), "Chưa có /content/dataset/det: chạy ô chép dữ liệu ở 1.5"
    out = RUNS / tag / "metrics"
    out.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(weights))
    imgs = list_images(LOCAL_DET / split / "images")

    # ---- 1. Chỉ số theo loại quả ----
    m = model.val(data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=BATCH,
                  project=str(RUNS / tag), name=f"val_{split}", exist_ok=True, plots=True, verbose=False)
    n_gt = np.zeros(ND, int)
    for lp in (LOCAL_DET / split / "labels").glob("*.txt"):
        for b in read_yolo_label(lp, nc=ND)[0]:
            n_gt[b[0]] += 1
    pos = {int(c): i for i, c in enumerate(m.box.ap_class_index)}
    rows = []
    for c in range(ND):
        p, r, ap50, ap = m.box.class_result(pos[c]) if c in pos else (np.nan,) * 4
        warn = ("KHÔNG CÓ MẪU" if n_gt[c] == 0 else
                f"ÍT MẪU ({n_gt[c]} < {MIN_BOX_EVAL}): số liệu không ổn định" if n_gt[c] < MIN_BOX_EVAL else "")
        rows.append(dict(class_name=FRUITS[c], n_box=int(n_gt[c]), precision=p, recall=r, mAP50=ap50, mAP50_95=ap, canh_bao=warn))
    per_class = pd.DataFrame(rows)
    overall = dict(tag=tag, split=split, weights=str(weights), fingerprint=fingerprint(p.name for p in imgs),
                   n_images=len(imgs), n_box=int(n_gt.sum()), precision=float(m.box.mp), recall=float(m.box.mr),
                   mAP50=float(m.box.map50), mAP50_95=float(m.box.map))

    # ---- 2. Confusion matrix (hàng = dự đoán, cột = nhãn thật, chỉ số ND = nền) ----
    M = np.array(m.confusion_matrix.matrix, dtype=float)
    # Kiểm tra chiều: tổng mỗi cột "nhãn thật" phải bằng số box thật của class đó
    if not np.allclose(M[:, :ND].sum(0), n_gt) and np.allclose(M[:ND, :].sum(1), n_gt):
        M = M.T
    elif not np.allclose(M[:, :ND].sum(0), n_gt):
        print("⚠️  Tổng cột confusion matrix khác số box thật; giả định hàng=dự đoán, cột=nhãn thật.")
    labels = FRUITS + ["background"]
    cm_df = pd.DataFrame(M.astype(int), index=[f"pred_{n}" for n in labels], columns=[f"true_{n}" for n in labels])
    summ = [("nhầm loại quả (tổng)", sum(M[p, t] for p in range(ND) for t in range(ND) if p != t)),
            ("bỏ sót (box thật -> nền)", M[ND, :ND].sum()),
            ("báo nhầm (nền -> có quả)", M[:ND, ND].sum())]
    summ += [(f"{FRUITS[t]} dự đoán thành {FRUITS[p]}", M[p, t]) for t in range(ND) for p in range(ND) if p != t]
    conf_summary = pd.DataFrame(summ, columns=["loai", "so_luong"])
    conf_summary["so_luong"] = conf_summary.so_luong.astype(int)

    # ---- 3. Ảnh sai / độ tin cậy thấp ----
    recs = []
    for res in model.predict([str(p) for p in imgs], conf=CONF_PRED, imgsz=IMGSZ, stream=True, verbose=False):
        h, w = res.orig_shape
        p = Path(res.path)
        gt = yolo_to_xyxy(read_yolo_label(LOCAL_DET / split / "labels" / (p.stem + ".txt"), nc=ND)[0], w, h)
        b = res.boxes
        pred = [(int(c), float(s), *xy) for c, s, xy in zip(b.cls.tolist(), b.conf.tolist(), b.xyxy.tolist())]
        mt = match_boxes([g[1:] for g in gt], [q[2:] for q in pred])
        wrong = sum(gt[i][0] != pred[j][0] for i, j in mt)
        low = sum(gt[i][0] == pred[j][0] and pred[j][1] < LOW_CONF for i, j in mt)
        fn, fp = len(gt) - len(mt), len(pred) - len(mt)
        recs.append(dict(image=p.name, source=source_of(p.name), n_gt=len(gt), n_pred=len(pred), fn=fn, fp=fp,
                         wrong_fruit=wrong, low_conf=low, min_conf=min([pred[j][1] for _, j in mt], default=1.0),
                         score=fn + fp + 2 * wrong, gt=gt, pred=pred))
    err = pd.DataFrame(recs)
    pick = pd.concat([err[err.score > 0].sort_values("score", ascending=False),
                      err[(err.score == 0) & (err.low_conf > 0)].sort_values("min_conf")]).head(N_ERROR_IMAGES)
    err_dir = out / f"errors_{split}"
    shutil.rmtree(err_dir, ignore_errors=True)
    err_dir.mkdir(parents=True)
    drawn = []
    for k, r in enumerate(pick.itertuples()):
        img = cv2.imread(str(LOCAL_DET / split / "images" / r.image))
        for c, *xy in r.gt:
            draw_box(img, xy, "GT:" + FRUITS[c], (0, 200, 0))
        for c, s, *xy in r.pred:
            draw_box(img, xy, f"{FRUITS[c]} {s:.2f}", (0, 0, 255))
        dst = err_dir / f"{k:02d}_{r.image}"
        cv2.imwrite(str(dst), img)
        drawn.append(dst)
    err_cols = ["image", "source", "n_gt", "n_pred", "fn", "fp", "wrong_fruit", "low_conf", "min_conf", "score"]
    err[err_cols].sort_values("score", ascending=False).to_csv(out / f"error_images_{split}.csv", index=False)

    # ---- 4. Tốc độ ----
    ms = time_per_item(lambda s: model.predict(s, imgsz=IMGSZ, verbose=False), [str(p) for p in imgs[:N_SPEED_IMAGES]])
    speed = dict(gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
                 n_images=min(len(imgs), N_SPEED_IMAGES), end_to_end_ms_per_img=ms,
                 **{f"val_{k}_ms": v for k, v in m.speed.items()})
    overall.update(speed_ms_per_img=ms, fruit_confusion=int(summ[0][1]))

    # ---- Lưu + hiển thị ----
    per_class.to_csv(out / f"per_class_{split}.csv", index=False)
    cm_df.to_csv(out / f"confusion_matrix_{split}.csv")
    conf_summary.to_csv(out / f"confusion_summary_{split}.csv", index=False)
    pd.DataFrame([speed]).to_csv(out / f"speed_{split}.csv", index=False)
    (out / f"summary_{split}.json").write_text(json.dumps(overall, indent=2, ensure_ascii=False))
    print(f"=== {tag} | {split} | {overall['n_images']} ảnh, {overall['n_box']} box ===")
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in overall.items() if k not in ("weights", "fingerprint")})
    display(per_class.round(4)); display(conf_summary)
    for r in per_class.itertuples():
        if r.canh_bao:
            print(f"⚠️  {r.class_name}: {r.canh_bao}")
    plot_confusion(M, labels, labels, f"Detection – {tag} ({split})", out / f"confusion_matrix_{split}.png",
                   ylabel="Dự đoán", xlabel="Nhãn thật")
    print("Số ảnh theo loại lỗi:", {k: int((err[k] > 0).sum()) for k in ("fn", "fp", "wrong_fruit", "low_conf")})
    show_grid(drawn, ncols=4, title=f"{tag}: ảnh sai / độ tin cậy thấp (xanh = nhãn thật, đỏ = dự đoán)", cell=4)
    print(f"Tốc độ: {ms:.2f} ms/ảnh (batch=1) | val speed: {m.speed}")
    print("Đã lưu bảng vào", out)
    return dict(per_class=per_class, confusion=cm_df, confusion_summary=conf_summary, overall=overall,
                errors=err[err_cols], speed=speed)


print("Đã định nghĩa evaluate_det().")

### 3.3 `evaluate_cls()`: tầng 2
Dự đoán từng ảnh trong `eval/test_own` (crop tự quay, **test chính**) và `eval/test_public`:
accuracy, precision/recall/F1 cho `fresh` và `rotten`, bảng **theo từng loại quả** (vd chuối hỏng bị đoán tươi), confusion matrix 2×2
(hàng = nhãn thật, cột = dự đoán), `N_ERROR_IMAGES` crop sai (sai tự tin nhất trước) hoặc đúng nhưng độ tin cậy thấp, tốc độ ms/crop.

In [ ]:
def cls_images(set_name):
    return [p for st in STATES for p in list_images(LOCAL_CLS / "eval" / set_name / st)]


def evaluate_cls(weights, tag="cls_baseline", sets=("test_own", "test_public")):
    weights = Path(weights)
    assert weights.exists(), f"Không thấy {weights}"
    assert (LOCAL_CLS / "eval").exists(), "Chưa có /content/dataset/cls: chạy ô 1.6b"
    out = RUNS / tag / "metrics"
    out.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(weights))
    names = model.names                                    # {0: 'fresh', 1: 'rotten'} theo tên thư mục
    assert sorted(names.values()) == sorted(STATES), f"Class của model {names} khác {STATES}"
    results, summary = {}, dict(tag=tag, weights=str(weights))
    for sname in sets:
        paths = cls_images(sname)
        if not paths:
            print(f"⚠️  eval/{sname} rỗng: bỏ qua")
            continue
        y_true = [STATES.index(p.parent.name) for p in paths]
        y_pred, conf = [], []
        for r in model.predict([str(p) for p in paths], imgsz=CLS_IMGSZ, batch=CLS_BATCH, stream=True, verbose=False):
            y_pred.append(STATES.index(names[int(r.probs.top1)]))
            conf.append(float(r.probs.top1conf))
        df = pd.DataFrame(dict(image=[p.name for p in paths], path=[str(p) for p in paths],
                               fruit=[fruit_of_crop(p.name) for p in paths],
                               true=[STATES[i] for i in y_true], pred=[STATES[i] for i in y_pred], conf=conf))
        df["correct"] = df.true == df.pred

        # Theo tình trạng
        p_, r_, f_, sup = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
        per_class = pd.DataFrame(dict(state=STATES, n=sup, precision=p_, recall=r_, f1=f_))
        per_class["canh_bao"] = [f"ÍT MẪU ({n} < {MIN_CROP_EVAL})" if n < MIN_CROP_EVAL else "" for n in sup]
        # Theo loại quả
        rows = []
        for f in FRUITS:
            sub = df[df.fruit == f]
            n_fr, n_ro = int((sub.true == "fresh").sum()), int((sub.true == "rotten").sum())
            if len(sub):
                pf, rf, ff, _ = precision_recall_fscore_support(sub.true, sub.pred, labels=STATES, zero_division=0)
            else:
                pf = rf = ff = [np.nan, np.nan]
            warn = "; ".join(f"{st}: {n} < {MIN_CROP_EVAL}" for st, n in (("fresh", n_fr), ("rotten", n_ro)) if n < MIN_CROP_EVAL)
            rows.append(dict(fruit=f, n=len(sub), n_fresh=n_fr, n_rotten=n_ro,
                             accuracy=sub.correct.mean() if len(sub) else np.nan,
                             f1_macro=np.nanmean(ff) if len(sub) else np.nan,
                             recall_rotten=rf[1], precision_rotten=pf[1],
                             fresh_doan_rotten=int(((sub.true == "fresh") & (sub.pred == "rotten")).sum()),
                             rotten_doan_fresh=int(((sub.true == "rotten") & (sub.pred == "fresh")).sum()),
                             canh_bao=("ÍT MẪU " + warn) if warn else ""))
        per_fruit = pd.DataFrame(rows)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])           # hàng = nhãn thật, cột = dự đoán
        ov = dict(set=sname, n=len(df), fingerprint=fingerprint(df.image), accuracy=float(df.correct.mean()),
                  f1_macro=float(np.mean(f_)), recall_rotten=float(r_[1]), precision_rotten=float(p_[1]))

        # Crop sai (sai tự tin nhất trước), rồi crop đúng nhưng độ tin cậy thấp
        pick = pd.concat([df[~df.correct].sort_values("conf", ascending=False),
                          df[df.correct & (df.conf < LOW_CONF)].sort_values("conf")]).head(N_ERROR_IMAGES)
        err_dir = out / f"cls_errors_{sname}"
        shutil.rmtree(err_dir, ignore_errors=True)
        err_dir.mkdir(parents=True)
        for k, r in enumerate(pick.itertuples()):
            shutil.copyfile(r.path, err_dir / f"{k:02d}_{r.true}_as_{r.pred}_{r.image}")

        df.drop(columns="path").to_csv(out / f"cls_predictions_{sname}.csv", index=False)
        per_class.to_csv(out / f"cls_per_class_{sname}.csv", index=False)
        per_fruit.to_csv(out / f"cls_per_fruit_{sname}.csv", index=False)
        pd.DataFrame(cm, index=[f"true_{s}" for s in STATES], columns=[f"pred_{s}" for s in STATES]).to_csv(out / f"cls_confusion_{sname}.csv")
        print(f"=== {tag} | {sname} | {len(df)} ảnh ===")
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in ov.items() if k != "fingerprint"})
        display(per_class.round(4)); display(per_fruit.round(4))
        for r in per_fruit.itertuples():
            if r.canh_bao:
                print(f"⚠️  {r.fruit}: {r.canh_bao}: số liệu không ổn định")
        plot_confusion(cm, STATES, STATES, f"Classifier – {tag} ({sname})", out / f"cls_confusion_{sname}.png",
                       ylabel="Nhãn thật", xlabel="Dự đoán")
        show_grid(pick.path, ncols=5, cell=2.6, title=f"{tag} / {sname}: crop sai hoặc độ tin cậy thấp",
                  captions=[f"{r.fruit}: {r.true}→{r.pred} {r.conf:.2f}" for r in pick.itertuples()])
        results[sname] = dict(per_class=per_class, per_fruit=per_fruit, confusion=cm, overall=ov, predictions=df)
        summary[sname] = ov

    sample = [p for s in sets for p in cls_images(s)][:N_SPEED_IMAGES]
    if sample:
        ms = time_per_item(lambda s: model.predict(s, imgsz=CLS_IMGSZ, verbose=False), [str(p) for p in sample])
        summary["speed_ms_per_crop"] = ms
        print(f"Tốc độ classifier: {ms:.2f} ms/crop (batch=1)")
    (out / "cls_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
    print("Đã lưu bảng vào", out)
    results["summary"] = summary
    return results


print("Đã định nghĩa evaluate_cls().")

### 3.4 Pipeline hai tầng và `evaluate_e2e()`
`pipeline_predict()` là toàn bộ suy luận: detect 3 loại quả → cắt crop (nới `CROP_PAD`, giống lúc tạo dữ liệu) → phân loại fresh/rotten.

`evaluate_e2e()` chạy pipeline trên **ảnh tự quay thuộc test** (nhãn 6 class), ghép box theo IoU >= `IOU_MATCH`, rồi tính:
- confusion matrix 6 class + nền (hàng = dự đoán, cột = nhãn thật); P/R/F1 theo từng class;
- **nhầm loại quả** (lỗi tầng 1) và **nhầm tình trạng** (cùng loại quả, khác fresh/rotten: lỗi tầng 2), bỏ sót, báo nhầm;
- **sai số đếm** mỗi ảnh: MAE theo loại quả và theo 6 class, tỷ lệ ảnh đếm đúng hoàn toàn;
- `N_ERROR_IMAGES` ảnh lỗi nhiều nhất, tốc độ ms/ảnh cho cả pipeline.

In [ ]:
def pipeline_predict(det_model, cls_model, img_bgr):
    """Tầng 1 (detect) -> crop -> Tầng 2 (fresh/rotten). Trả về list dict: fruit, state, det_conf, cls_conf, xyxy."""
    r = det_model.predict(img_bgr, conf=CONF_PRED, imgsz=IMGSZ, verbose=False)[0]
    xyxy, fr, dc = r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()
    if not xyxy:
        return []
    crops = [crop_box(img_bgr, b) for b in xyxy]
    rs = cls_model.predict(crops, imgsz=CLS_IMGSZ, verbose=False)
    return [dict(fruit=FRUITS[int(f)], state=cls_model.names[int(p.probs.top1)], det_conf=float(c),
                 cls_conf=float(p.probs.top1conf), xyxy=b) for b, f, c, p in zip(xyxy, fr, dc, rs)]


def own_test_table():
    """Ảnh tự quay thuộc test và có nhãn 6 class."""
    si = pd.read_csv(SPLIT_IMAGES)
    man = pd.read_csv(MERGED / "manifest.csv")
    t = man[(man.origin == "own") & (man.has_state.astype(str) == "True")]
    return t[t.merged_name.isin(set(si.image[si.split == "test"]))].reset_index(drop=True)


def evaluate_e2e(det_w, cls_w, tag="e2e_baseline"):
    t = own_test_table()
    assert len(t), "Chưa có ảnh tự quay (nhãn 6 class) thuộc test: không đánh giá end-to-end được"
    out = RUNS / tag / "metrics"
    out.mkdir(parents=True, exist_ok=True)
    own_names = load_names(find_data_yaml(OWN))
    det, cls = YOLO(str(det_w)), YOLO(str(cls_w))
    M = np.zeros((NC + 1, NC + 1))                 # hàng = dự đoán, cột = nhãn thật, chỉ số NC = nền
    recs, counts = [], []
    for r in t.itertuples():
        img = cv2.imread(str(LOCAL_DET / "test" / "images" / r.merged_name))
        h, w = img.shape[:2]
        boxes, _ = read_yolo_label(label_path_for(Path(r.orig_image)), nc=len(own_names))
        gt = [(CLASS_TO_ID[own_names[c]], *xy) for c, *xy in yolo_to_xyxy(boxes, w, h)]
        pr = [(class6(d["fruit"], d["state"]), d["det_conf"], d["cls_conf"], *d["xyxy"]) for d in pipeline_predict(det, cls, img)]
        mt = match_boxes([g[1:] for g in gt], [q[3:] for q in pr])
        mg, mp = {i for i, _ in mt}, {j for _, j in mt}
        for i, j in mt:
            M[pr[j][0], gt[i][0]] += 1
        for j in set(range(len(pr))) - mp:
            M[pr[j][0], NC] += 1
        for i in set(range(len(gt))) - mg:
            M[NC, gt[i][0]] += 1
        wf = sum(CLASS_FRUIT[pr[j][0]] != CLASS_FRUIT[gt[i][0]] for i, j in mt)
        ws = sum(CLASS_FRUIT[pr[j][0]] == CLASS_FRUIT[gt[i][0]] and pr[j][0] != gt[i][0] for i, j in mt)
        fn, fp = len(gt) - len(mt), len(pr) - len(mt)
        recs.append(dict(image=r.merged_name, source=r.source, n_gt=len(gt), n_pred=len(pr), fn=fn, fp=fp,
                         wrong_fruit=wf, wrong_state=ws, score=fn + fp + 2 * (wf + ws), gt=gt, pred=pr))
        row = dict(image=r.merged_name)
        for f in FRUITS:
            row[f"gt_{f}"] = sum(CLASS_FRUIT[g[0]] == f for g in gt)
            row[f"pred_{f}"] = sum(CLASS_FRUIT[q[0]] == f for q in pr)
        for c in range(NC):
            row[f"gt_{CLASS_NAMES[c]}"] = sum(g[0] == c for g in gt)
            row[f"pred_{CLASS_NAMES[c]}"] = sum(q[0] == c for q in pr)
        counts.append(row)

    # ---- Chỉ số theo 6 class ----
    rows = []
    for c in range(NC):
        n_true, n_pred, tp = M[:, c].sum(), M[c, :].sum(), M[c, c]
        p = tp / n_pred if n_pred else np.nan
        rc = tp / n_true if n_true else np.nan
        rows.append(dict(class_name=CLASS_NAMES[c], n_box=int(n_true), precision=p, recall=rc,
                         f1=2 * p * rc / (p + rc) if n_true and n_pred and (p + rc) else (0.0 if n_true else np.nan),
                         canh_bao="KHÔNG CÓ MẪU" if n_true == 0 else (f"ÍT MẪU ({int(n_true)} < {MIN_BOX_EVAL})" if n_true < MIN_BOX_EVAL else "")))
    per_class = pd.DataFrame(rows)
    n_gt_all = M[:, :NC].sum()
    matched = M[:NC, :NC].sum()
    same_fruit = sum(M[p, q] for p in range(NC) for q in range(NC) if CLASS_FRUIT[p] == CLASS_FRUIT[q])
    ci = CLASS_TO_ID
    apples, oranges = [ci["apple_fresh"], ci["apple_rotten"]], [ci["orange_fresh"], ci["orange_rotten"]]
    conf_summary = pd.DataFrame([
        ("đúng hoàn toàn (đúng loại quả + tình trạng)", np.trace(M[:NC, :NC])),
        ("nhầm loại quả (lỗi tầng 1)", matched - same_fruit),
        ("nhầm tình trạng (cùng loại quả, lỗi tầng 2)", same_fruit - np.trace(M[:NC, :NC])),
        ("bỏ sót (box thật -> nền)", M[NC, :NC].sum()),
        ("báo nhầm (nền -> có quả)", M[:NC, NC].sum()),
        ("theo dõi: banana_fresh -> banana_rotten (vd đốm nâu chuối chín)", M[ci["banana_rotten"], ci["banana_fresh"]]),
        ("theo dõi: apple_rotten -> apple_fresh (vd vết thâm nhỏ)", M[ci["apple_fresh"], ci["apple_rotten"]]),
        ("theo dõi: apple_rotten bị bỏ sót (-> nền)", M[NC, ci["apple_rotten"]]),
        ("theo dõi: cam dự đoán thành táo", sum(M[p, q] for p in apples for q in oranges)),
    ], columns=["loai", "so_luong"])
    conf_summary["so_luong"] = conf_summary.so_luong.astype(int)
    has = per_class[per_class.n_box > 0]
    overall = dict(tag=tag, det_weights=str(det_w), cls_weights=str(cls_w), fingerprint=fingerprint(t.merged_name),
                   n_images=len(t), n_box=int(n_gt_all),
                   e2e_accuracy=float(np.trace(M[:NC, :NC]) / n_gt_all) if n_gt_all else np.nan,   # box thật được đoán đúng 6 class
                   detection_recall=float(matched / n_gt_all) if n_gt_all else np.nan,            # box thật được phát hiện (bất kể class)
                   state_acc_given_fruit=float(np.trace(M[:NC, :NC]) / same_fruit) if same_fruit else np.nan,
                   f1_macro=float(has.f1.mean()) if len(has) else np.nan)

    # ---- Sai số đếm ----
    cnt = pd.DataFrame(counts)
    crow = []
    for k in FRUITS + CLASS_NAMES:
        d = cnt[f"pred_{k}"] - cnt[f"gt_{k}"]
        crow.append(dict(muc=k, tong_that=int(cnt[f"gt_{k}"].sum()), tong_du_doan=int(cnt[f"pred_{k}"].sum()),
                         MAE_moi_anh=float(d.abs().mean()), lech_trung_binh=float(d.mean())))
    counting = pd.DataFrame(crow)
    exact = (cnt[[f"pred_{f}" for f in FRUITS]].to_numpy() == cnt[[f"gt_{f}" for f in FRUITS]].to_numpy()).all(1)
    overall.update(count_MAE_per_fruit=float(counting.MAE_moi_anh[:ND].mean()), count_exact_rate=float(exact.mean()))

    # ---- Ảnh lỗi ----
    err = pd.DataFrame(recs)
    pick = err[err.score > 0].sort_values("score", ascending=False).head(N_ERROR_IMAGES)
    err_dir = out / "errors_e2e"
    shutil.rmtree(err_dir, ignore_errors=True)
    err_dir.mkdir(parents=True)
    drawn = []
    for k, r in enumerate(pick.itertuples()):
        img = cv2.imread(str(LOCAL_DET / "test" / "images" / r.image))
        for c, *xy in r.gt:
            draw_box(img, xy, "GT:" + CLASS_NAMES[c], (0, 200, 0))
        for c, dcf, ccf, *xy in r.pred:
            draw_box(img, xy, f"{CLASS_NAMES[c]} d{dcf:.2f} c{ccf:.2f}", (0, 0, 255))
        dst = err_dir / f"{k:02d}_{r.image}"
        cv2.imwrite(str(dst), img)
        drawn.append(dst)

    # ---- Tốc độ cả pipeline ----
    ms = time_per_item(lambda p: pipeline_predict(det, cls, cv2.imread(p)),
                       [str(LOCAL_DET / "test" / "images" / n) for n in t.merged_name[:N_SPEED_IMAGES]])
    overall["speed_ms_per_img"] = ms

    # ---- Lưu + hiển thị ----
    labels = CLASS_NAMES + ["background"]
    pd.DataFrame(M.astype(int), index=[f"pred_{n}" for n in labels], columns=[f"true_{n}" for n in labels]).to_csv(out / "e2e_confusion.csv")
    per_class.to_csv(out / "e2e_per_class.csv", index=False)
    conf_summary.to_csv(out / "e2e_confusion_summary.csv", index=False)
    counting.to_csv(out / "e2e_counting.csv", index=False)
    cnt.to_csv(out / "e2e_counts_per_image.csv", index=False)
    err.drop(columns=["gt", "pred"]).to_csv(out / "e2e_error_images.csv", index=False)
    (out / "e2e_summary.json").write_text(json.dumps(overall, indent=2, ensure_ascii=False))
    print(f"=== {tag} | {len(t)} ảnh tự quay (test), {int(n_gt_all)} box ===")
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in overall.items() if "weights" not in k and k != "fingerprint"})
    display(per_class.round(4)); display(conf_summary); display(counting.round(3))
    for r in per_class.itertuples():
        if r.canh_bao:
            print(f"⚠️  {r.class_name}: {r.canh_bao}")
    plot_confusion(M, labels, labels, f"End-to-end – {tag}", out / "e2e_confusion.png", ylabel="Dự đoán", xlabel="Nhãn thật")
    show_grid(drawn, ncols=4, cell=4, title=f"{tag}: ảnh lỗi nhiều nhất (xanh = nhãn thật, đỏ = dự đoán; d = conf detect, c = conf classifier)")
    print(f"Tốc độ pipeline: {ms:.2f} ms/ảnh (gồm đọc ảnh, detect, crop, phân loại)")
    print("Đã lưu bảng vào", out)
    return dict(per_class=per_class, confusion_summary=conf_summary, counting=counting, overall=overall)


print("Đã định nghĩa pipeline_predict() và evaluate_e2e().")

### 3.5 Đánh giá baseline
Mỗi ô chạy độc lập (sau 3.1–3.4); bỏ qua ô nào chưa có weights tương ứng.

In [ ]:
res_det_base = evaluate_det(RUNS / "det_baseline" / "weights" / "best.pt", split="test", tag="det_baseline")
res_det_base_sub = {sp: evaluate_det(RUNS / "det_baseline" / "weights" / "best.pt", split=sp, tag="det_baseline")
                    for sp in ("test_public", "test_own") if (LOCAL_DET / sp / "images").exists()}

In [ ]:
res_cls_base = evaluate_cls(RUNS / "cls_baseline" / "weights" / "best.pt", tag="cls_baseline")

In [ ]:
try:
    res_e2e_base = evaluate_e2e(RUNS / "det_baseline" / "weights" / "best.pt", RUNS / "cls_baseline" / "weights" / "best.pt",
                                tag="e2e_baseline")
except AssertionError as e:
    print(f"[BỎ QUA VÌ CHƯA ĐỦ ĐIỀU KIỆN]: {e}")

### 3.6 Ghi chú phân tích lỗi (bạn tự điền sau khi xem ảnh)
Chỉ ghi những gì **quan sát được** trên ảnh/crop lỗi (`runs/*/metrics/errors_*`, `cls_errors_*`), không suy đoán nguyên nhân khi chưa có bằng chứng.
Các bảng `confusion_summary` cho số lượng thật của từng cặp cần theo dõi; cột "Tầng" giúp chỉ ra lỗi nằm ở đâu.

| Nhóm dễ nhầm cần kiểm tra | Tầng | Số ảnh/crop quan sát thấy | Ghi chú |
|---|---|---|---|
| Đốm nâu tự nhiên của chuối chín bị đoán `rotten` | 2 | | |
| Vết thâm nhỏ trên táo bị bỏ sót (`apple_rotten` → `fresh`) | 2 | | |
| Cam xanh nhầm thành táo xanh | 1 | | |
| Quả chồng nhau hoặc bị che (bỏ sót box, hoặc crop lẫn quả khác) | 1 / 2 | | |
| Crop bị mờ / cắt lệch làm classifier sai | 2 | | |
| Nhóm khác quan sát được | | | |

Class/tổ hợp ít mẫu trong test: ghi rõ số mẫu kèm số liệu, không gộp vào trung bình để che đi.

## Phần 4. Fine-tune bằng dữ liệu tự quay
Hai tầng fine-tune độc lập. Train = dữ liệu tự quay thuộc tập train **cộng** `REPLAY_RATIO` × dữ liệu public thuộc train (lấy mẫu theo SEED) để không quên
kiến thức cũ. **Val/test giữ nguyên**; ảnh/crop test **không bao giờ** vào train (có `assert`).

### 4.1 Dữ liệu fine-tune detection → `work/data_ft.yaml`

In [ ]:
si = pd.read_csv(SPLIT_IMAGES)
own_train = sorted(si.image[(si.split == "train") & (si.origin == "own")])
pub_train = sorted(si.image[(si.split == "train") & (si.origin == "public")])
assert own_train, "Chưa có ảnh tự quay (own-*) trong tập train: gán nhãn frame, đặt vào own/ rồi chạy lại Phần 1"
replay = sorted(random.Random(SEED).sample(pub_train, round(REPLAY_RATIO * len(pub_train))))
ft_names = own_train + replay
print(f"det train_ft = {len(own_train)} ảnh tự quay + {len(replay)} ảnh public replay ({REPLAY_RATIO:.0%} của {len(pub_train)})")

# --- Chống rò rỉ: train_ft không chạm val/test (theo ảnh VÀ theo nhóm) ---
assert (LOCAL_DET / "test").exists(), "Chưa có /content/dataset/det: chạy ô chép dữ liệu ở 1.5"
grp = dict(zip(si.image, si.group))
ft_groups = {grp[n] for n in ft_names}
for sp in ("val", "test"):
    sp_names = {p.name for p in list_images(LOCAL_DET / sp / "images")}
    assert sp_names == set(si.image[si.split == sp]), f"/content/dataset/det/{sp} không khớp {SPLIT_IMAGES.name}: chạy lại ô 1.5"
    assert not (set(ft_names) & sp_names), f"RÒ RỈ ảnh giữa train_ft và {sp}"
    assert not (ft_groups & {grp[n] for n in sp_names}), f"RÒ RỈ nhóm ảnh giữa train_ft và {sp}"
print("✅ Không rò rỉ ảnh/nhóm giữa det train_ft và val/test.")

build_local_split("train_ft", ft_names)
ft_cfg = {
    "path": str(LOCAL_DET),
    "train": str(LOCAL_DET / "train_ft" / "images"),
    "val": str(LOCAL_DET / "val" / "images"),
    "test": str(LOCAL_DET / "test" / "images"),
    "nc": ND,
    "names": {i: n for i, n in enumerate(FRUITS)},
}
write_text_guarded(DATA_FT_YAML, yaml.safe_dump(ft_cfg, sort_keys=False, allow_unicode=True), "split")
print(DATA_FT_YAML.read_text())

### 4.2 Dữ liệu fine-tune classifier → `/content/dataset/cls/ft`
- `ft/train` = crop tự quay `own/train` + replay `REPLAY_RATIO` × ảnh `base/train`.
- `ft/val` = `base/val` + crop tự quay `own/val` (để chọn model tốt trên cả miền ảnh thật).

In [ ]:
assert (LOCAL_CLS / "own" / "train").exists(), "Chưa có /content/dataset/cls: chạy ô 1.6b"
own_tr = [p for st in STATES for p in list_images(LOCAL_CLS / "own" / "train" / st)]
pub_tr = [p for st in STATES for p in list_images(LOCAL_CLS / "base" / "train" / st)]
assert own_tr, "Chưa có crop tự quay thuộc train (cần ảnh own/ gán nhãn 6 class)"
replay = random.Random(SEED).sample(pub_tr, round(REPLAY_RATIO * len(pub_tr)))
val = [p for s in ("base/val", "own/val") for st in STATES for p in list_images(LOCAL_CLS / s / st)]

ft = fresh_local_dir(LOCAL_CLS / "ft")
def _put(args):
    p, split = args
    dst = ft / split / p.parent.name / p.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(p, dst)
for st in STATES:
    (ft / "train" / st).mkdir(parents=True, exist_ok=True)
    (ft / "val" / st).mkdir(parents=True, exist_ok=True)
parallel_map(_put, [(p, "train") for p in own_tr + replay] + [(p, "val") for p in val])
print(f"cls ft/train = {len(own_tr)} crop tự quay + {len(replay)} ảnh public replay; ft/val = {len(val)} ảnh")

# --- Chống rò rỉ: ft không chạm test (theo tên crop, theo nguồn tự quay, theo nhóm ảnh public) ---
ft_names = {p.name for p in list_images(ft)}
test_own = {p.name for p in list_images(LOCAL_CLS / "eval" / "test_own")}
test_pub = {p.name for p in list_images(LOCAL_CLS / "eval" / "test_public")}
assert not (ft_names & (test_own | test_pub)), "RÒ RỈ: crop test nằm trong dữ liệu fine-tune"
if OWN_SPLIT_MODE == "video":
    ft_own_src = {source_of(n) for n in ft_names if is_own_source(source_of(n))}
    assert not (ft_own_src & {source_of(n) for n in test_own}), "RÒ RỈ: nguồn tự quay của test nằm trong fine-tune"
assert all((LOCAL_CLS / "base" / "train" / p.parent.name / p.name).exists() for p in replay), "Replay phải lấy từ base/train"
print("✅ Không rò rỉ giữa cls ft và eval/test_own, eval/test_public.")
display(pd.crosstab([fruit_of_crop(p.name) for p in list_images(ft / "train")],
                    [p.parent.name for p in list_images(ft / "train")]).rename_axis(index="fruit", columns="state"))

### 4.3 Fine-tune detection (ô nặng)
Khởi tạo từ `runs/det_baseline/weights/best.pt`, `lr0=LR0_FT`, tuỳ chọn đóng băng `FREEZE_LAYERS` lớp đầu (`FREEZE_FT`).

In [ ]:
base_best = RUNS / "det_baseline" / "weights" / "best.pt"
assert base_best.exists(), f"Chưa có {base_best}: train baseline detection trước"
run_train("det_finetune", base_best, data=str(DATA_FT_YAML), epochs=EPOCHS_FT, imgsz=IMGSZ, batch=BATCH,
          optimizer=OPTIMIZER, lr0=LR0_FT, freeze=FREEZE_LAYERS if FREEZE_FT else None)

### 4.3b Train JOINT detection (khuyến nghị)
Một lần train duy nhất trên ảnh public + ảnh tự quay nhân bản `OWN_REP` lần, khởi tạo từ COCO `JOINT_MODEL`. Fine-tune ở 4.3 chọn `best.pt` ở epoch 1 vì val chủ yếu là public; joint cho mAP test cao hơn. Kết quả: `runs/det_joint`.

In [ ]:
pub_imgs = sorted(list_images(LOCAL_DET / "train" / "images"))
own_imgs = sorted(p for p in list_images(LOCAL_DET / "train_ft" / "images") if p.name.startswith("own-"))
assert own_imgs, "Chưa có train_ft: chạy ô 4.1 trước"
joint_list = WORK / "det_joint_train.txt"
joint_list.write_text("\n".join(str(p) for p in pub_imgs + own_imgs * OWN_REP) + "\n")
joint_cfg = {"path": str(LOCAL_DET), "train": str(joint_list), "val": str(LOCAL_DET / "val" / "images"),
             "test": str(LOCAL_DET / "test" / "images"), "nc": ND, "names": {i: n for i, n in enumerate(FRUITS)}}
DATA_JOINT_YAML = WORK / "data_joint.yaml"
DATA_JOINT_YAML.write_text(yaml.safe_dump(joint_cfg, sort_keys=False, allow_unicode=True))
print(f"joint train: {len(pub_imgs)} public + {len(own_imgs)} own x{OWN_REP}")
run_train("det_joint", f"{JOINT_MODEL}.pt", data=str(DATA_JOINT_YAML), epochs=EPOCHS_JOINT, imgsz=IMGSZ, batch=BATCH,
          optimizer=OPTIMIZER, lr0=LR0_BASE)


In [ ]:
# Đánh giá det_joint trên test (đừng chạy khi đang train, 6GB sẽ OOM)
res_det_joint = evaluate_det(RUNS / "det_joint" / "weights" / "best.pt", split="test", tag="det_joint")


### 4.4 Fine-tune classifier (ô nặng)

In [ ]:
cls_best = RUNS / "cls_baseline" / "weights" / "best.pt"
assert cls_best.exists(), f"Chưa có {cls_best}: train baseline classifier trước"
run_train("cls_finetune", cls_best, data=str(LOCAL_CLS / "ft"), epochs=EPOCHS_CLS_FT, imgsz=CLS_IMGSZ,
          batch=CLS_BATCH, optimizer=CLS_OPTIMIZER, lr0=LR0_CLS_FT)

### 4.5 Tiếp tục fine-tune khi bị ngắt (CHỈ chạy khi cần)
Phiên mới → Phần 0 → 1.5 → 1.6b → 2.0 → 4.1 / 4.2 (tạo lại dữ liệu fine-tune) →
đặt `RESUME['det_finetune']` / `RESUME['cls_finetune']` = True, chạy lại CONFIG → chạy ô này.

In [ ]:
run_resume("det_finetune", LOCAL_DET / "train_ft")
run_resume("cls_finetune", LOCAL_CLS / "ft" / "train")

### 4.6 Đường cong fine-tune

In [ ]:
hist_det_ft = show_training(RUNS / "det_finetune")
hist_cls_ft = show_training(RUNS / "cls_finetune")

### 4.7 So sánh trước/sau: tầng 1 (detection) trên CÙNG tập test
Nếu kết quả baseline trong phiên hiện tại không có hoặc được tính trên tập test khác (mã băm khác), ô sẽ đánh giá lại baseline.

In [ ]:
def compare(base, ft, key, metrics, keep=()):
    """Ghép bảng baseline và fine-tune theo `key`, thêm cột chênh lệch <metric>_delta."""
    df = base[[key, *keep, *metrics]].merge(ft[[key, *metrics]], on=key, suffixes=("_base", "_ft"))
    for k in metrics:
        df[f"{k}_delta"] = df[f"{k}_ft"] - df[f"{k}_base"]
    return df


def bar_before_after(ax, labels, base_vals, ft_vals, title, ylabel):
    x = np.arange(len(labels))
    ax.bar(x - 0.2, base_vals, 0.4, label="baseline")
    ax.bar(x + 0.2, ft_vals, 0.4, label="finetune")
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylim(0, 1); ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(axis="y", alpha=.3); ax.legend()

In [ ]:
res_det_ft = evaluate_det(RUNS / "det_finetune" / "weights" / "best.pt", split="test", tag="det_finetune")
fp_ = res_det_ft["overall"]["fingerprint"]
rb = globals().get("res_det_base")
if rb is None or rb["overall"]["fingerprint"] != fp_:
    print("Chưa có kết quả baseline trên tập test hiện tại -> đánh giá lại baseline.")
    rb = res_det_base = evaluate_det(RUNS / "det_baseline" / "weights" / "best.pt", split="test", tag="det_baseline")
assert rb["overall"]["fingerprint"] == fp_, "Baseline và fine-tune không cùng tập test"

M_DET = ["precision", "recall", "mAP50", "mAP50_95"]
cmp_det = compare(rb["per_class"], res_det_ft["per_class"], "class_name", M_DET, keep=("n_box",))
cmp_det_all = pd.DataFrame([{"model": "baseline", **{k: rb["overall"][k] for k in M_DET + ["speed_ms_per_img"]}},
                            {"model": "finetune", **{k: res_det_ft["overall"][k] for k in M_DET + ["speed_ms_per_img"]}}])
out = RUNS / "det_finetune" / "metrics"
cmp_det.to_csv(out / "compare_per_class_test.csv", index=False)
cmp_det_all.to_csv(out / "compare_overall_test.csv", index=False)
display(cmp_det_all.round(4)); display(cmp_det.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bar_before_after(axes[0], [f"{n}\n(n={b})" for n, b in zip(cmp_det.class_name, cmp_det.n_box)],
                 cmp_det.mAP50_base, cmp_det.mAP50_ft, "Detection mAP@0.5 theo loại quả (n = số box test)", "mAP@0.5")
bar_before_after(axes[1], ["mAP@0.5", "mAP@0.5:0.95"], cmp_det_all.loc[0, ["mAP50", "mAP50_95"]].astype(float),
                 cmp_det_all.loc[1, ["mAP50", "mAP50_95"]].astype(float), "Detection: toàn bộ", "mAP")
plt.tight_layout(); fig.savefig(out / "compare_map_test.png", dpi=150); plt.show()

# --- Cùng so sánh trên test CHỈ public (so được với baseline cũ) và test CHỈ tự quay ---
rows = []
for sp in ("test_public", "test_own"):
    if not (LOCAL_DET / sp / "images").exists():
        continue
    rb_sp = globals().get("res_det_base_sub", {}).get(sp) or evaluate_det(RUNS / "det_baseline" / "weights" / "best.pt", split=sp, tag="det_baseline")
    rf_sp = evaluate_det(RUNS / "det_finetune" / "weights" / "best.pt", split=sp, tag="det_finetune")
    assert rb_sp["overall"]["fingerprint"] == rf_sp["overall"]["fingerprint"], f"{sp}: baseline và fine-tune không cùng tập"
    for name, r in (("baseline", rb_sp), ("finetune", rf_sp)):
        rows.append({"split": sp, "model": name, "n_box": r["overall"]["n_box"], **{k: r["overall"][k] for k in M_DET}})
cmp_sub = pd.DataFrame(rows)
cmp_sub.to_csv(RUNS / "det_finetune" / "metrics" / "compare_by_origin_test.csv", index=False)
display(cmp_sub.round(4))

### 4.8 So sánh trước/sau: tầng 2 (classifier)

In [ ]:
res_cls_ft = evaluate_cls(RUNS / "cls_finetune" / "weights" / "best.pt", tag="cls_finetune")
rb = globals().get("res_cls_base")
same = rb is not None and all(rb.get(s, {}).get("overall", {}).get("fingerprint") == res_cls_ft[s]["overall"]["fingerprint"]
                              for s in res_cls_ft if s != "summary")
if not same:
    print("Chưa có kết quả baseline trên cùng tập test -> đánh giá lại baseline classifier.")
    rb = res_cls_base = evaluate_cls(RUNS / "cls_baseline" / "weights" / "best.pt", tag="cls_baseline")

M_CLS = ["accuracy", "f1_macro", "recall_rotten", "precision_rotten"]
out = RUNS / "cls_finetune" / "metrics"
sets = [s for s in res_cls_ft if s != "summary"]
fig, axes = plt.subplots(1, len(sets), figsize=(6.5 * len(sets), 4), squeeze=False)
for ax, s in zip(axes[0], sets):
    assert rb[s]["overall"]["fingerprint"] == res_cls_ft[s]["overall"]["fingerprint"], f"{s}: baseline và fine-tune không cùng tập"
    ov = pd.DataFrame([{"model": "baseline", **{k: rb[s]["overall"][k] for k in M_CLS}},
                       {"model": "finetune", **{k: res_cls_ft[s]["overall"][k] for k in M_CLS}}])
    pf = compare(rb[s]["per_fruit"], res_cls_ft[s]["per_fruit"], "fruit", M_CLS[:3], keep=("n", "n_fresh", "n_rotten"))
    ov.to_csv(out / f"compare_cls_overall_{s}.csv", index=False)
    pf.to_csv(out / f"compare_cls_per_fruit_{s}.csv", index=False)
    print(f"--- {s} ---"); display(ov.round(4)); display(pf.round(4))
    bar_before_after(ax, [f"{f}\n(n={n})" for f, n in zip(pf.fruit, pf.n)], pf.accuracy_base, pf.accuracy_ft,
                     f"Classifier accuracy theo loại quả – {s}", "accuracy")
plt.tight_layout(); fig.savefig(out / "compare_cls.png", dpi=150); plt.show()

### 4.9 So sánh trước/sau: end-to-end trên ảnh tự quay
Baseline = (det_baseline + cls_baseline), fine-tune = (det_finetune + cls_finetune). Kết hợp với 4.7, 4.8 để chỉ ra thay đổi đến từ tầng nào.

In [ ]:
W = lambda run: RUNS / run / "weights" / "best.pt"
res_e2e_ft = evaluate_e2e(W("det_finetune"), W("cls_finetune"), tag="e2e_finetune")
rb = globals().get("res_e2e_base")
if rb is None or rb["overall"]["fingerprint"] != res_e2e_ft["overall"]["fingerprint"]:
    print("Chưa có kết quả baseline trên cùng tập ảnh -> đánh giá lại end-to-end baseline.")
    rb = res_e2e_base = evaluate_e2e(W("det_baseline"), W("cls_baseline"), tag="e2e_baseline")

M_E2E = ["e2e_accuracy", "detection_recall", "state_acc_given_fruit", "f1_macro", "count_MAE_per_fruit", "count_exact_rate", "speed_ms_per_img"]
out = RUNS / "e2e_finetune" / "metrics"
ov = pd.DataFrame([{"model": "baseline", **{k: rb["overall"][k] for k in M_E2E}},
                   {"model": "finetune", **{k: res_e2e_ft["overall"][k] for k in M_E2E}}])
pc = compare(rb["per_class"], res_e2e_ft["per_class"], "class_name", ["precision", "recall", "f1"], keep=("n_box",))
cs = rb["confusion_summary"].merge(res_e2e_ft["confusion_summary"], on="loai", suffixes=("_base", "_ft"))
cnt = compare(rb["counting"], res_e2e_ft["counting"], "muc", ["MAE_moi_anh"], keep=("tong_that",))
for name, df in (("overall", ov), ("per_class", pc), ("confusion_summary", cs), ("counting", cnt)):
    df.to_csv(out / f"compare_e2e_{name}.csv", index=False)
    display(df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw=dict(width_ratios=[2.2, 1]))
bar_before_after(axes[0], [f"{n}\n(n={b})" for n, b in zip(pc.class_name, pc.n_box)], pc.f1_base, pc.f1_ft,
                 "End-to-end F1 theo 6 class (n = số box test)", "F1")
bar_before_after(axes[1], ["e2e acc", "det recall", "state acc"], ov.loc[0, M_E2E[:3]].astype(float),
                 ov.loc[1, M_E2E[:3]].astype(float), "End-to-end: toàn bộ", "tỷ lệ")
plt.tight_layout(); fig.savefig(out / "compare_e2e.png", dpi=150); plt.show()

## Phần 5. Xuất model
### 5.1 Chép `best.pt` vào `exports/` và xuất ONNX
Bốn model: detection và classifier, mỗi tầng một bản baseline và một bản fine-tune. Khi xuất ONNX, Ultralytics có thể tự cài gói `onnx` (phụ thuộc của chính nó).

In [ ]:
from ultralytics import YOLO

EXPORTS.mkdir(parents=True, exist_ok=True)
for run, sz in (("det_baseline", IMGSZ), ("det_finetune", IMGSZ), ("det_joint", IMGSZ), ("cls_baseline", CLS_IMGSZ), ("cls_finetune", CLS_IMGSZ)):
    src = RUNS / run / "weights" / "best.pt"
    if not src.exists():
        print(f"[BỎ QUA] chưa có {src}")
        continue
    dst = EXPORTS / f"{run}_best.pt"
    shutil.copyfile(src, dst)
    onnx_path = Path(YOLO(str(dst)).export(format="onnx", imgsz=sz))   # tạo exports/<run>_best.onnx
    for f in (dst, onnx_path):
        print(f"{f.name:28s} {f.stat().st_size / 1e6:8.2f} MB  -> {f}")

### 5.2 Tải file về máy
Trình duyệt có thể hỏi cho phép tải nhiều file. Có thể tải trực tiếp từ Google Drive (`exports/`) thay cho ô này.

In [ ]:
for f in sorted(EXPORTS.glob("*_best.*")):
    if IN_COLAB:
        from google.colab import files
        print("Tải", f.name)
        files.download(str(f))
    else:
        print("Local: model nằm ở", f)

### 5.3 Phiên bản thư viện và cấu hình suy luận
Cài **đúng các phiên bản này** trên máy local để chạy `best.pt`/ONNX cho kết quả giống Colab.
`inference_config.json` lưu các tham số pipeline (`CONF_PRED`, `CROP_PAD`, kích thước ảnh, tên class) để dùng lại `pipeline_predict` ở máy local.

In [ ]:
import platform, torch, ultralytics, cv2
vers = {"python": platform.python_version(), "ultralytics": ultralytics.__version__, "torch": torch.__version__,
        "opencv": cv2.__version__, "numpy": np.__version__, "pandas": pd.__version__}
try:
    import onnx
    vers["onnx"] = onnx.__version__
except ImportError:
    pass
for k, v in vers.items():
    print(f"{k:12s} {v}")
print("\nLệnh cài trên máy local:")
print(f"pip install ultralytics=={vers['ultralytics']} torch=={vers['torch'].split('+')[0]}")
(EXPORTS / "versions.json").write_text(json.dumps(vers, indent=2))
(EXPORTS / "inference_config.json").write_text(json.dumps(dict(
    fruits=FRUITS, states=STATES, class_names=CLASS_NAMES, imgsz=IMGSZ, cls_imgsz=CLS_IMGSZ,
    conf_pred=CONF_PRED, crop_pad=CROP_PAD), indent=2))
print("Đã ghi", EXPORTS / "versions.json", "và", EXPORTS / "inference_config.json")